<a href="https://colab.research.google.com/github/sandip156/speaker-disjoint/blob/main/sandip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi


Wed Oct 14 07:26:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | grep "Tesla"


Mounted at /content/drive
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |


In [ ]:
!pip install -q nemo_toolkit[asr] jiwer
import nemo.collections.asr as nemo_asr
print("NeMo ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/1

NeMo ready


In [ ]:
import json, os
from pathlib import Path

# load their published SOTA model (downloads automatically)
model = nemo_asr.models.ASRModel.from_pretrained("ilprl-docse/NwachaMuna-NepConformer-Aug")
print("model loaded")

# locate your test set on Drive
MANIFEST = "/content/drive/MyDrive/experiments/test_v1_16k.json"
rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8")]
print(f"test utterances: {len(rows)}")   # expect 197

# transcribe
texts = model.transcribe([r["audio_filepath"] for r in rows], batch_size=16)
hyps = texts[0] if isinstance(texts, tuple) else texts   # NeMo version differences

NwachaMuna-NepConformer-Aug.nemo: reconstructing file:   0%|          |  0.00B /  123MB            

NwachaMuna-NepConformer-Aug.nemo: downloading bytes:           |  0.00B            

[NeMo I 2026-10-16 15:25:11 mixins:194] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-10-16 15:25:11 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_train.json
    sample_rate: 16000
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-10-16 15:25:11 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_val.json
    sample_rate: 16000
    batch_size: 4
    shuffle: false
    use

[NeMo I 2026-10-16 15:25:12 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--ilprl-docse--NwachaMuna-NepConformer-Aug/snapshots/eba3d554805ae275fba9f0cda6374e15c94b0e7f/NwachaMuna-NepConformer-Aug.nemo.
model loaded


[NeMo W 2026-10-16 15:25:14 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-16 15:25:14 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


test utterances: 127


Transcribing: 0it [00:01, ?it/s]


AudioLoadingError: Fetching info about audio from '/Users/sandipjoshi/Desktop/experiments/audio_16k/audio_59.wav' failed. Details:
Exception #0 (<class 'lhotse.audio.backend.LibsndfileBackend'>): <class 'soundfile.LibsndfileError'>: Error opening '/Users/sandipjoshi/Desktop/experiments/audio_16k/audio_59.wav': System error.
Exception #1 (<class 'lhotse.audio.backend.TorchcodecBackend'>): <class 'RuntimeError'>: SingleStreamDecoder, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:51, Could not open input file: /Users/sandipjoshi/Desktop/experiments/audio_16k/audio_59.wav No such file or directory
Exception #2 (<class 'lhotse.audio.backend.TorchaudioDefaultBackend'>): <class 'AttributeError'>: module 'torchaudio' has no attribute 'info'
Exception #3 (<class 'lhotse.audio.backend.AudioreadBackend'>): <class 'FileNotFoundError'>: [Errno 2] No such file or directory: '/Users/sandipjoshi/Desktop/experiments/audio_16k/audio_59.wav'
Set LHOTSE_AU

In [ ]:
import json
from pathlib import Path, PurePosixPath

model = nemo_asr.models.ASRModel.from_pretrained("ilprl-docse/NwachaMuna-NepConformer-Aug")
print("model loaded")

MANIFEST = "/content/drive/MyDrive/experiments/test_spk08_16k.json"
rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8")]

# rewrite Mac paths -> Colab Drive paths (basename approach: robust)
for r in rows:
    r["audio_filepath"] = "/content/drive/MyDrive/experiments/audio_16k/" + Path(r["audio_filepath"]).name

# sanity check BEFORE transcribing: every file must exist
missing = [r["audio_filepath"] for r in rows if not Path(r["audio_filepath"]).exists()]
print(f"test utterances: {len(rows)} | missing files: {len(missing)}")
assert not missing, f"still missing: {missing[:5]}"

texts = model.transcribe([r["audio_filepath"] for r in rows], batch_size=16)
hyps = texts[0] if isinstance(texts, tuple) else texts
print("transcription done")

[NeMo I 2026-10-16 15:25:39 mixins:194] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-10-16 15:25:39 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_train.json
    sample_rate: 16000
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-10-16 15:25:39 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_val.json
    sample_rate: 16000
    batch_size: 4
    shuffle: false
    use

[NeMo I 2026-10-16 15:25:40 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--ilprl-docse--NwachaMuna-NepConformer-Aug/snapshots/eba3d554805ae275fba9f0cda6374e15c94b0e7f/NwachaMuna-NepConformer-Aug.nemo.
model loaded


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/experiments/test_spk08_16k.json'

In [ ]:
import re, unicodedata, jiwer

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[।?.,!;:]", "", s)              # danda + punctuation
    s = s.replace("\u200c", "").replace("\u200d", "")  # ZWJ/ZWNJ
    s = re.sub(r"\s+", " ", s).strip()            # collapse spaces
    return s

refs  = [norm(r["text"]) for r in rows]
hyps_n = [norm(h) for h in hyps]

cer = jiwer.cer(refs, hyps_n)
wer = jiwer.wer(refs, hyps_n)
print(f"\n=== E2 HEADLINE ===")
print(f"their published model -> your unseen speaker (spk08)")
print(f"CER: {cer*100:.2f}%   WER: {wer*100:.2f}%   (n=197)")

# per-utterance results -> Drive (feeds E5 later)
import csv
with open("/content/drive/MyDrive/experiments/e2_results.tsv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["id","domain","reference","hypothesis","cer"])
    for r, ref, h in zip(rows, refs, hyps_n):
        from jiwer import cer as c1
        w.writerow([r.get("id",""), r.get("domain",""), ref, h, round(c1([ref],[h]),4)])
print("saved: e2_results.tsv")

NameError: name 'hyps' is not defined

In [ ]:
import re, unicodedata, jiwer, csv

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[।?.,!;:]", "", s)
    s = s.replace("\u200c", "").replace("\u200d", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) extract the REAL text from hypotheses (NeMo 2.x returns objects)
def hyp_text(h):
    if isinstance(h, str): return h
    if hasattr(h, "text"): return h.text
    if hasattr(h, "pred_text"): return h.pred_text
    return str(h)

hyps_txt = [hyp_text(h) for h in hyps]
refs_txt = [norm(r["text"]) for r in rows]

# 2) sanity print BEFORE scoring — eyes on actual data, always
for i in range(3):
    print(f"REF [{i}]: {refs_txt[i][:80]}")
    print(f"HYP [{i}]: {hyps_txt[i][:80]}\n")

assert all(refs_txt), "EMPTY REFERENCE found — stop and inspect"
assert all(len(t) < 500 for t in hyps_txt), "hypothesis absurdly long — inspect samples above"

cer = jiwer.cer(refs_txt, [norm(h) for h in hyps_txt])
wer = jiwer.wer(refs_txt, [norm(h) for h in hyps_txt])
print(f"=== E2 HEADLINE (corrected) ===")
print(f"their published SOTA -> your unseen speaker spk08, n=197")
print(f"CER: {cer*100:.2f}%   WER: {wer*100:.2f}%")

# 3) per-utterance results -> Drive (overwrites the garbage version)
with open("/content/drive/MyDrive/experiments/e2_results.tsv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["id","domain","reference","hypothesis","cer"])
    for r, ref, h in zip(rows, refs_txt, [norm(h) for h in hyps_txt]):
        w.writerow([r.get("id",""), r.get("domain",""), ref, h, round(jiwer.cer([ref],[h]),4)])
print("saved: e2_results.tsv")

NameError: name 'hyps' is not defined

In [ ]:
import csv, random
from collections import defaultdict

random.seed(42)
by_dom = defaultdict(lambda: ([], []))
for r, ref, h in zip(rows, refs_txt, [norm(x) for x in hyps_txt]):
    by_dom[r["domain"]][0].append(ref); by_dom[r["domain"]][1].append(h)

print(f"{'domain':<26}{'n':>4}{'CER%':>8}   95%CI")
dom_rows = []
for d, (rs, hs) in sorted(by_dom.items()):
    base = jiwer.cer(rs, hs)
    boots = []
    for _ in range(1000):
        idx = [random.randrange(len(rs)) for _ in rs]
        boots.append(jiwer.cer([rs[i] for i in idx], [hs[i] for i in idx]))
    lo, hi = sorted(boots)[24], sorted(boots)[975]
    dom_rows.append([d, len(rs), round(base*100,2), round(lo*100,2), round(hi*100,2)])
    print(f"{d:<26}{len(rs):>4}{base*100:>8.2f}   [{lo*100:.1f}, {hi*100:.1f}]")

with open("/content/drive/MyDrive/experiments/e5_domain_results.tsv","w",encoding="utf-8",newline="") as f:
    w = csv.writer(f, delimiter="\t"); w.writerow(["domain","n","cer","ci_lo","ci_hi"]); w.writerows(dom_rows)
print("saved: e5_domain_results.tsv")

NameError: name 'refs_txt' is not defined

In [ ]:
import statistics
durs = [r["duration"] for r in rows]
cers = [jiwer.cer([ref],[h]) for ref, h in zip(refs_txt, [norm(x) for x in hyps_txt])]
long_half  = [c for d, c in zip(durs, cers) if d >= statistics.median(durs)]
short_half = [c for d, c in zip(durs, cers) if d <  statistics.median(durs)]
print(f"shorter-half utterances: CER {sum(short_half)/len(short_half)*100:.1f}%  (n={len(short_half)})")
print(f"longer-half  utterances: CER {sum(long_half)/len(long_half)*100:.1f}%  (n={len(long_half)})")

shorter-half utterances: CER 25.9%  (n=98)
longer-half  utterances: CER 27.8%  (n=99)


In [ ]:
import json
from pathlib import Path

MANIFEST = "/content/drive/MyDrive/experiments/test_v1_16k.json"
rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8")]

# rewrite Mac paths -> Colab Drive paths
for r in rows:
    r["audio_filepath"] = "/content/drive/MyDrive/experiments/audio_16k/" + Path(r["audio_filepath"]).name

missing = [r["audio_filepath"] for r in rows if not Path(r["audio_filepath"]).exists()]
print(f"test utterances: {len(rows)} | missing files: {len(missing)}")   # expect: 127 | 0
assert not missing, f"missing: {missing[:5]}"

texts = model.transcribe([r["audio_filepath"] for r in rows], batch_size=16)
hyps = texts[0] if isinstance(texts, tuple) else texts
print("transcription done")

[NeMo W 2026-10-16 15:27:33 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-16 15:27:33 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


test utterances: 127 | missing files: 0


Transcribing: 8it [01:55, 14.41s/it]

transcription done


In [ ]:
import re, unicodedata, jiwer, csv

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[।?.,!;:]", "", s)
    s = s.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r"\s+", " ", s).strip()

def hyp_text(h):
    if isinstance(h, str): return h
    if hasattr(h, "text"): return h.text
    if hasattr(h, "pred_text"): return h.pred_text
    return str(h)

hyps_txt = [hyp_text(h) for h in hyps]
refs_txt = [norm(r["text"]) for r in rows]

for i in range(3):
    print(f"REF [{i}]: {refs_txt[i][:80]}")
    print(f"HYP [{i}]: {hyps_txt[i][:80]}\n")
assert all(refs_txt), "EMPTY REFERENCE"
assert all(len(t) < 500 for t in hyps_txt), "hypothesis absurdly long"

cer = jiwer.cer(refs_txt, [norm(h) for h in hyps_txt])
wer = jiwer.wer(refs_txt, [norm(h) for h in hyps_txt])
print(f"=== E2 HEADLINE v1.0 ===")
print(f"their published SOTA -> unseen speaker spk04, n={len(rows)}")
print(f"CER: {cer*100:.2f}%   WER: {wer*100:.2f}%")

with open("/content/drive/MyDrive/experiments/e2_results_v1.tsv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["id","speaker","domain","reference","hypothesis","cer"])
    for r, ref, h in zip(rows, refs_txt, [norm(x) for x in hyps_txt]):
        w.writerow([r.get("id",""), r.get("speaker",""), r.get("domain",""), ref, h,
                    round(jiwer.cer([ref],[h]),4)])
print("saved: e2_results_v1.tsv")

REF [0]: ये स्वनिगलय स्वयेबह गु छु छु थाय्यात दु
HYP [0]: य स्वनेगलय् स्वयेबहःगु छु छु छायात दु

REF [1]: थ्व बस गु गु थासय दिकी
HYP [1]: थ्व भस गुकु थासय् दिकि

REF [2]: थ्व होटल गुगल म्यापय बांलाःगु रेटिङ का गु दु
HYP [2]: थ्वतेनं गुंगलमयपे बांलाकु रेटुंकागु दु

=== E2 HEADLINE v1.0 ===
their published SOTA -> unseen speaker spk04, n=127
CER: 26.75%   WER: 73.62%
saved: e2_results_v1.tsv


In [ ]:
import csv, random
from collections import defaultdict

random.seed(42)
by_dom = defaultdict(lambda: ([], []))
for r, ref, h in zip(rows, refs_txt, [norm(x) for x in hyps_txt]):
    by_dom[r["domain"]][0].append(ref); by_dom[r["domain"]][1].append(h)

print(f"{'domain':<26}{'n':>4}{'CER%':>8}   95%CI")
dom_rows = []
for d, (rs, hs) in sorted(by_dom.items()):
    base = jiwer.cer(rs, hs)
    boots = []
    for _ in range(1000):
        idx = [random.randrange(len(rs)) for _ in rs]
        boots.append(jiwer.cer([rs[i] for i in idx], [hs[i] for i in idx]))
    lo, hi = sorted(boots)[24], sorted(boots)[975]
    dom_rows.append([d, len(rs), round(base*100,2), round(lo*100,2), round(hi*100,2)])
    print(f"{d:<26}{len(rs):>4}{base*100:>8.2f}   [{lo*100:.1f}, {hi*100:.1f}]")

with open("/content/drive/MyDrive/experiments/e5_domain_results_v1.tsv","w",encoding="utf-8",newline="") as f:
    w = csv.writer(f, delimiter="\t"); w.writerow(["domain","n","cer","ci_lo","ci_hi"]); w.writerows(dom_rows)
print("saved: e5_domain_results_v1.tsv")

domain                       n    CER%   95%CI
animals_nature              12   15.34   [7.2, 23.6]
culture_festivals            1   20.00   [20.0, 20.0]
directions_navigation        7   31.90   [23.3, 42.7]
everyday_objects             4   22.62   [19.0, 27.2]
family_kinship               6   19.17   [11.6, 26.2]
food_cuisine                 9   23.40   [18.1, 30.7]
food_restaurants            12   28.03   [23.4, 33.8]
geography_places             3   28.57   [25.0, 30.8]
guides_services             12   31.60   [25.9, 38.9]
health_body                 10   19.08   [14.1, 25.6]
hotels_accommodation        14   29.50   [23.9, 33.9]
shopping_local_markets       9   27.93   [21.1, 35.0]
tourist_places_culture      15   26.92   [22.9, 31.0]
travel_conversation          6   20.13   [16.9, 24.8]
travel_transportation        7   34.45   [30.1, 37.9]
saved: e5_domain_results_v1.tsv


In [ ]:
import json, os
from pathlib import Path, PurePosixPath
from omegaconf import OmegaConf
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
import nemo.collections.asr as nemo_asr

# fix Drive paths in manifests (same trick as before)
def fix_and_load(name):
    p = f"/content/drive/MyDrive/experiments/{name}"
    rows = [json.loads(l) for l in open(p, encoding="utf-8")]
    for r in rows:
        r["audio_filepath"] = "/content/drive/MyDrive/experiments/audio_16k/" + Path(r["audio_filepath"]).name
    out = f"/content/{name}"
    with open(out, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return out, len(rows)

train_m, n_tr = fix_and_load("train_v1_274.json")
val_m,   n_va = fix_and_load("val_v1_30.json")
print(f"train: {n_tr} | val: {n_va}")

model = nemo_asr.models.ASRModel.from_pretrained("ilprl-docse/NwachaMuna-NepConformer-Aug")

model.encoder.freeze()                     # their paper: frozen ≈ full FT, safer at 0.33h
model.setup_optimization(OmegaConf.create({"name": "adamw", "lr": 1e-5,
                                           "weight_decay": 1e-3}))
model.setup_training_data(OmegaConf.create({
    "manifest_filepath": train_m, "sample_rate": 16000, "batch_size": 4,
    "shuffle": True, "num_workers": 2, "max_duration": 16.7, "min_duration": 0.1}))
model.setup_validation_data(OmegaConf.create({
    "manifest_filepath": val_m, "sample_rate": 16000, "batch_size": 4,
    "shuffle": False, "num_workers": 2}))

ckpt_cb = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/experiments/finetune_ckpts",
    filename="ft_{epoch}-{val_wer:.4f}", monitor="val_wer", mode="min", save_top_k=2)

trainer = pl.Trainer(devices=1, accelerator="gpu", max_epochs=60,
                     precision="16-mixed", check_val_every_n_epoch=2,
                     log_every_n_steps=10, callbacks=[ckpt_cb],
                     default_root_dir="/content/drive/MyDrive/experiments/finetune")
trainer.fit(model)
print("BEST:", ckpt_cb.best_model_path, ckpt_cb.best_model_score)

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
!pip install -q nemo_toolkit[asr] jiwer pytorch_lightning
import nemo.collections.asr as nemo_asr
import pytorch_lightning as pl
from omegaconf import OmegaConf
print("environment ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.9/903.9 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8

environment ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint

In [ ]:
import json
from pathlib import Path
from omegaconf import OmegaConf
import lightning.pytorch as pl                          # <-- NEW namespace
from lightning.pytorch.callbacks import ModelCheckpoint # <-- NEW namespace
import nemo.collections.asr as nemo_asr

# fix Drive paths in manifests
def fix_and_load(name):
    p = f"/content/drive/MyDrive/experiments/{name}"
    rows = [json.loads(l) for l in open(p, encoding="utf-8")]
    for r in rows:
        r["audio_filepath"] = "/content/drive/MyDrive/experiments/audio_16k/" + Path(r["audio_filepath"]).name
    out = f"/content/{name}"
    with open(out, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return out, len(rows)

train_m, n_tr = fix_and_load("train_v1_274.json")
val_m,   n_va = fix_and_load("val_v1_30.json")
print(f"train: {n_tr} | val: {n_va}")

model = nemo_asr.models.ASRModel.from_pretrained("ilprl-docse/NwachaMuna-NepConformer-Aug")

model.encoder.freeze()
model.setup_optimization(OmegaConf.create({"name": "adamw", "lr": 1e-5, "weight_decay": 1e-3}))
model.setup_training_data(OmegaConf.create({
    "manifest_filepath": train_m, "sample_rate": 16000, "batch_size": 4,
    "shuffle": True, "num_workers": 2, "max_duration": 16.7, "min_duration": 0.1}))
model.setup_validation_data(OmegaConf.create({
    "manifest_filepath": val_m, "sample_rate": 16000, "batch_size": 4,
    "shuffle": False, "num_workers": 2}))

ckpt_cb = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/experiments/finetune_ckpts",
    filename="ft_{epoch}-{val_wer:.4f}", monitor="val_wer", mode="min", save_top_k=2)

trainer = pl.Trainer(devices=1, accelerator="gpu", max_epochs=60,
                     precision="16-mixed", check_val_every_n_epoch=2,
                     log_every_n_steps=10, callbacks=[ckpt_cb],
                     default_root_dir="/content/drive/MyDrive/experiments/finetune")
trainer.fit(model)
print("BEST:", ckpt_cb.best_model_path, ckpt_cb.best_model_score)

train: 274 | val: 30
[NeMo I 2026-10-15 09:51:17 mixins:194] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-10-15 09:51:17 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_train.json
    sample_rate: 16000
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-10-15 09:51:17 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_val.json
    sample_rate: 16000
    batch_size: 4
    shuffle: false
    use

[NeMo I 2026-10-15 09:51:17 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--ilprl-docse--NwachaMuna-NepConformer-Aug/snapshots/eba3d554805ae275fba9f0cda6374e15c94b0e7f/NwachaMuna-NepConformer-Aug.nemo.


[NeMo W 2026-10-15 09:51:17 modelPT:669] Trainer wasn't specified in model constructor. Make sure that you really wanted it.


[NeMo I 2026-10-15 09:51:17 modelPT:781] Optimizer config = AdamW (
    Parameter Group 0
        amsgrad: False
        betas: (0.9, 0.999)
        capturable: False
        decoupled_weight_decay: True
        differentiable: False
        eps: 1e-08
        foreach: None
        fused: None
        lr: 1e-05
        maximize: False
        weight_decay: 0.001
    )
[NeMo I 2026-10-15 09:51:17 lr_scheduler:844] Scheduler not initialized as no `sched` config supplied to setup_optimizer()
[NeMo I 2026-10-15 09:51:17 collections:201] Dataset loaded with 274 files totalling 0.33 hours
[NeMo I 2026-10-15 09:51:17 collections:202] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-10-15 09:51:17 collections:201] Dataset loaded with 30 files totalling 0.03 hours
[NeMo I 2026-10-15 09:51:17 collections:202] 0 files were filtered totalling 0.00 hours


...

[NeMo I 2026-10-15 09:58:28 wer:318] 
    
[NeMo I 2026-10-15 09:58:28 wer:319] WER reference:बानेश्वरनिंसे कोटेश्वर सतिक लाः।
[NeMo I 2026-10-15 09:58:28 wer:320] WER predicted:बानेस्वलिसे कोोतेस्वर संतीक ला
[NeMo I 2026-10-15 09:58:29 wer:318] 
    
[NeMo I 2026-10-15 09:58:29 wer:319] WER reference:सिम न्यायेत पासपोर्ट व फोटो माः ।
[NeMo I 2026-10-15 09:58:29 wer:320] WER predicted:सिव ल्यायात पासपोर्ट वा फोद माः
[NeMo I 2026-10-15 09:58:29 wer:318] 
    
[NeMo I 2026-10-15 09:58:29 wer:319] WER reference:स्वन्ति वयेत लच्छि मानि ।
[NeMo I 2026-10-15 09:58:29 wer:320] WER predicted:सन्ते वयात लछि मानि
[NeMo I 2026-10-15 09:58:29 wer:318] 
    
[NeMo I 2026-10-15 09:58:29 wer:319] WER reference:होटेल्या लोकेसन म्यापय सेभ यानाः तइगु दु।
[NeMo I 2026-10-15 09:58:29 wer:320] WER predicted:भोटेंलया लुकि सनन्या पिया स्े यानाःतयगु दु
[NeMo I 2026-10-15 09:58:29 wer:318] 
    
[NeMo I 2026-10-15 09:58:29 wer:319] WER reference:पैंया पुन्हि बाछि लिपा मोहनी वइ ।
[NeMo I 2026-10-15 09:58:29 wer

Validation: |          | 0/? [00:00<?, ?it/s]

[NeMo I 2026-10-15 09:58:42 wer:318] 
    
[NeMo I 2026-10-15 09:58:42 wer:319] WER reference:बानेश्वरनिंसे कोटेश्वर सतिक लाः।
[NeMo I 2026-10-15 09:58:42 wer:320] WER predicted:बानेस्वलिसे कोोतेस्वर संतीक ला
[NeMo I 2026-10-15 09:58:42 wer:318] 
    
[NeMo I 2026-10-15 09:58:42 wer:319] WER reference:सिम न्यायेत पासपोर्ट व फोटो माः ।
[NeMo I 2026-10-15 09:58:42 wer:320] WER predicted:सिव ल्यायात पासपोर्ट वा फोद माः
[NeMo I 2026-10-15 09:58:42 wer:318] 
    
[NeMo I 2026-10-15 09:58:42 wer:319] WER reference:स्वन्ति वयेत लच्छि मानि ।
[NeMo I 2026-10-15 09:58:42 wer:320] WER predicted:सन्ते वयात लछि मानि
[NeMo I 2026-10-15 09:58:42 wer:318] 
    
[NeMo I 2026-10-15 09:58:42 wer:319] WER reference:होटेल्या लोकेसन म्यापय सेभ यानाः तइगु दु।
[NeMo I 2026-10-15 09:58:42 wer:320] WER predicted:भोटेंलया लुकि सनन्या पिया स्े यानाःतयगु दु
[NeMo I 2026-10-15 09:58:42 wer:318] 
    
[NeMo I 2026-10-15 09:58:42 wer:319] WER reference:पैंया पुन्हि बाछि लिपा मोहनी वइ ।
[NeMo I 2026-10-15 09:58:42 wer

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=60` reached.


BEST: /content/drive/MyDrive/experiments/finetune_ckpts/ft_epoch=59-val_wer=0.3110.ckpt tensor(0.3110, device='cuda:0')


In [ ]:
import json
from pathlib import Path

best = model   # fine-tuned weights already in memory from trainer.fit
print("using the in-memory fine-tuned model")

rows = [json.loads(l) for l in open("/content/drive/MyDrive/experiments/test_v1_16k.json", encoding="utf-8")]
for r in rows:
    r["audio_filepath"] = "/content/drive/MyDrive/experiments/audio_16k/" + Path(r["audio_filepath"]).name

missing = [r["audio_filepath"] for r in rows if not Path(r["audio_filepath"]).exists()]
print(f"test utterances: {len(rows)} | missing: {len(missing)}")
assert not missing

texts = best.transcribe([r["audio_filepath"] for r in rows], batch_size=16)
hyps = texts[0] if isinstance(texts, tuple) else texts
print("transcription done")

using the in-memory fine-tuned model


[NeMo W 2026-10-15 10:05:25 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-15 10:05:25 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


test utterances: 127 | missing: 0


Transcribing: 8it [01:31, 11.42s/it]

transcription done


In [ ]:
import re, unicodedata, jiwer, csv

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[।?.,!;:]", "", s)
    s = s.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r"\s+", " ", s).strip()

def hyp_text(h):
    if isinstance(h, str): return h
    if hasattr(h, "text"): return h.text
    if hasattr(h, "pred_text"): return h.pred_text
    return str(h)

hyps_txt = [hyp_text(h) for h in hyps]
refs_txt = [norm(r["text"]) for r in rows]

for i in range(3):
    print(f"REF [{i}]: {refs_txt[i][:80]}")
    print(f"HYP [{i}]: {hyps_txt[i][:80]}\n")
assert all(refs_txt), "EMPTY REFERENCE"
assert all(len(t) < 500 for t in hyps_txt), "hypothesis absurdly long"

cer = jiwer.cer(refs_txt, [norm(h) for h in hyps_txt])
wer = jiwer.wer(refs_txt, [norm(h) for h in hyps_txt])
print(f"=== E3: fine-tuned on 304 train utts -> unseen speaker spk04, n={len(rows)} ===")
print(f"CER: {cer*100:.2f}%   WER: {wer*100:.2f}%")

with open("/content/drive/MyDrive/experiments/e3_results_v1.tsv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f, delimiter="\t")
    w.writerow(["id","speaker","domain","reference","hypothesis","cer"])
    for r, ref, h in zip(rows, refs_txt, [norm(x) for x in hyps_txt]):
        w.writerow([r.get("id",""), r.get("speaker",""), r.get("domain",""), ref, h,
                    round(jiwer.cer([ref],[h]),4)])
print("saved: e3_results_v1.tsv")

REF [0]: ये स्वनिगलय स्वयेबह गु छु छु थाय्यात दु
HYP [0]: य स्वनेगलय् स्वयेबहःगु छुछु  छा यात दु।

REF [1]: थ्व बस गु गु थासय दिकी
HYP [1]: थ्व वस गुकु थासय् दिकि।

REF [2]: थ्व होटल गुगल म्यापय बांलाःगु रेटिङ का गु दु
HYP [2]: थ्वटेनं गुँगलमयपे बांलाकु रइटुं कागु दु

=== E3: fine-tuned on 304 train utts -> unseen speaker spk04, n=127 ===
CER: 26.78%   WER: 73.62%
saved: e3_results_v1.tsv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | grep Tesla

Mounted at /content/drive
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |


In [ ]:
!pip install -q transformers accelerate bitsandbytes sacrebleu pandas
print("LLM stack ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.8 MB/s eta 0:00:00
LLM stack ready


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/experiments/llm_probe_30.tsv", sep="\t")
print(f"{len(df)} triples loaded")
print(df[["id","domain"]].head(10).to_string(index=False))
print("\nSample Newari:", df.newar_sentence.iloc[0][:60])
print("Sample Nepali:", df.nepali_sentence.iloc[0][:60])
print("Sample English:", df.english_sentence.iloc[0][:60])

30 triples loaded
 id                 domain
 58        guides_services
 13 shopping_local_markets
167        weather_climate
152      culture_festivals
141      culture_festivals
 98 tourist_places_culture
 53   hotels_accommodation
407  travel_transportation
 45  travel_transportation
430  directions_navigation

Sample Newari: गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?
Sample Nepali: गाइड साहेब, आज कति घण्टा हिँड्ने हो?
Sample English: Guide, how many hours will we walk today?


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config)
print(f"{MODEL_ID} loaded")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen/Qwen2.5-7B-Instruct loaded


In [ ]:
def ask(model, tokenizer, system, user):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.strip()

# quick smoke test — does the model even respond?
print(ask(model, tokenizer,
          "You are a translator.",
          "Translate this Newari sentence into Nepali. Output only the translation.\nSentence: गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?"))

गाइड साहेब, थौ गुलिघण्टा निशाचरण वनोंमा थिए?


In [ ]:
import pandas as pd, time

df = pd.read_csv("/content/drive/MyDrive/experiments/llm_probe_30.tsv", sep="\t")
SYSTEM = "You are a translator."
answers = []
start = time.time()
for i, row in df.iterrows():
    q = (f"Translate this Newari sentence into Nepali. Output only the translation.\n"
         f"Sentence: {row.newar_sentence}")
    ans = ask(model, tokenizer, SYSTEM, q)
    answers.append(ans)
    if (i+1) % 10 == 0:
        print(f"{i+1}/30 done ({time.time()-start:.0f}s)")

df["qwen_nepali"] = answers
df.to_csv("/content/drive/MyDrive/experiments/probe_qwen_task1.tsv",
          sep="\t", index=False)
print("saved: probe_qwen_task1.tsv")

10/30 done (48s)
20/30 done (95s)
30/30 done (144s)
saved: probe_qwen_task1.tsv


In [ ]:
for i in [0, 5, 10]:
    print(f"NEWARI : {df.newar_sentence.iloc[i][:70]}")
    print(f"TRUTH  : {df.nepali_sentence.iloc[i][:70]}")
    print(f"QWEN   : {df.qwen_nepali.iloc[i][:70]}\n")

NEWARI : गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?
TRUTH  : गाइड साहेब, आज कति घण्टा हिँड्ने हो?
QWEN   : गाइड साहेब, थौ गुलिघण्टा निशाचरण वनोंमा थिए?

NEWARI : स्वयम्भूया माकतय्सं मि्हचा थ्वके फु, सावधान जुयादिसा
TRUTH  : स्वयम्भूका बादरहरूले झोला खोस्न सक्छन्, सावधान रहनुहोस्।
QWEN   : स्वयम्भूया माकत्यां मिठाइको थवा फु, सावधान जुनाईसा

NEWARI : रूम सर्भिसय नसा छ्‌वया हयादिसँ।
TRUTH  : रुम सर्भिसमा खाना पठाइदिनुहोस्।
QWEN   : रूम सर्भिसय नसा छवा हुन्यादिन।



In [ ]:
import sacrebleu
df["chrf"] = [sacrebleu.corpus_chrf([h], [t]).score
              for h, t in zip(df.qwen_nepali, df.nepali_sentence)]
print(f"Qwen-2.5-7B — mean chrF vs verified Nepali: {df.chrf.mean():.1f}")
print(f"(reference scale: 0 = unrelated text, ~30-50 = decent translation)")

Qwen-2.5-7B — mean chrF vs verified Nepali: 8.8
(reference scale: 0 = unrelated text, ~30-50 = decent translation)


In [ ]:
import sacrebleu
df["chrf"] = [sacrebleu.corpus_chrf([h], [t]).score
              for h, t in zip(df.qwen_nepali, df.nepali_sentence)]
print(f"Qwen-2.5-7B — mean chrF vs verified Nepali: {df.chrf.mean():.1f}")
print(f"scale: 0 = unrelated, ~30-50 = decent translation, 60+ = good")
print(f"best row:  {df.loc[df.chrf.idxmax(), 'id']}  chrF {df.chrf.max():.1f}")
print(f"worst row: {df.loc[df.chrf.idxmin(), 'id']}  chrF {df.chrf.min():.1f}")

Qwen-2.5-7B — mean chrF vs verified Nepali: 8.8
scale: 0 = unrelated, ~30-50 = decent translation, 60+ = good
best row:  201  chrF 20.8
worst row: 13  chrF 0.0


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto",
    quantization_config=bnb_config)
print(f"{MODEL_ID} loaded")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

mistralai/Mistral-7B-Instruct-v0.3 loaded


In [ ]:
print(ask(model, tokenizer,
          "You are a translator.",
          "Translate this Newari sentence into Nepali. Output only the translation.\nSentence: गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?"))

गाइड साहेब, तिनी गुलि घण्टा रखिएको छैन जंगलमा छ?

Translation: Guide Sahib, are the bells not hung in the forest?


In [ ]:
answers = []
for i, row in df.iterrows():
    q = (f"Translate this Newari sentence into Nepali. Output only the translation.\n"
         f"Sentence: {row.newar_sentence}")
    answers.append(ask(model, tokenizer, "You are a translator.", q))
    if (i+1) % 10 == 0: print(f"{i+1}/30")

df["mistral_nepali"] = answers
df.to_csv("/content/drive/MyDrive/experiments/probe_mistral_task1.tsv",
          sep="\t", index=False)
print("saved: probe_mistral_task1.tsv")

10/30
20/30
30/30
saved: probe_mistral_task1.tsv


In [ ]:
for i in [0, 5, 10]:
    print(f"NEWARI : {df.newar_sentence.iloc[i][:70]}")
    print(f"TRUTH  : {df.nepali_sentence.iloc[i][:70]}")
    print(f"MISTRAL: {df.mistral_nepali.iloc[i][:70]}\n")

df["mistral_chrf"] = [sacrebleu.corpus_chrf([h], [t]).score
                      for h, t in zip(df.mistral_nepali, df.nepali_sentence)]
print(f"Mistral-7B  mean chrF: {df.mistral_chrf.mean():.1f}")
print(f"Qwen-2.5-7B mean chrF: 8.8")

NEWARI : गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?
TRUTH  : गाइड साहेब, आज कति घण्टा हिँड्ने हो?
MISTRAL: गाइड साहेब, तिनी गुलि घण्टा रखिएको छैन जंगलमा छ?

Translation: Guide S

NEWARI : स्वयम्भूया माकतय्सं मि्हचा थ्वके फु, सावधान जुयादिसा
TRUTH  : स्वयम्भूका बादरहरूले झोला खोस्न सक्छन्, सावधान रहनुहोस्।
MISTRAL: अहं मात्र माकतय मिहिचा थ्वके फु, सावधान जुन हुन्दोस्

This translates 

NEWARI : रूम सर्भिसय नसा छ्‌वया हयादिसँ।
TRUTH  : रुम सर्भिसमा खाना पठाइदिनुहोस्।
MISTRAL: घर सर्व समे छैन हुन्छन्। (Ghar sarv samē chain hun'chhan.)

This trans

Mistral-7B  mean chrF: 3.1
Qwen-2.5-7B mean chrF: 8.8


In [1]:
!pip install -q "nemo_toolkit[asr]" "huggingface_hub" 2>&1 | tail -5
print("install done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 36.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
install done


In [2]:
import json, zipfile, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/experiments')
AUDIO_DIR = Path('/content/audio_16k')

# --- audio: accepts zip OR loose folder, both inside Drive/experiments ---
zip_path = DRIVE_DIR / 'audio_16k.zip'
if AUDIO_DIR.exists() and len(list(AUDIO_DIR.glob('*.wav'))) > 0:
    print('audio already local, reusing')
elif zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content/')
    print('unzipped', zip_path.name)
elif (DRIVE_DIR / 'audio_16k').exists():
    print('copying audio folder from Drive to local disk (fast I/O)...')
    shutil.copytree(DRIVE_DIR / 'audio_16k', AUDIO_DIR, dirs_exist_ok=True)
else:
    raise SystemExit('No audio found. Expected audio_16k.zip or audio_16k/ inside Drive/experiments')

# repair: if the zip extracted flat (no folder prefix), move stray wavs
if len(list(AUDIO_DIR.glob('*.wav'))) == 0:
    stray = list(Path('/content').glob('audio_*.wav'))
    if stray:
        AUDIO_DIR.mkdir(exist_ok=True)
        for f in stray:
            shutil.move(str(f), str(AUDIO_DIR / f.name))
        print('moved', len(stray), 'stray wavs into audio_16k/')

print('audio files:', len(list(AUDIO_DIR.glob('*.wav'))), '(expect 431)')

# --- manifests ---
rows = []
need = ['test_v1_16k.json', 'train_v1_274.json', 'val_v1_30.json']
absent = [n for n in need if not (DRIVE_DIR / n).exists()]
if absent:
    raise SystemExit(f'Missing manifest(s) in Drive/experiments: {absent}')
for name in need:
    part = [json.loads(l) for l in open(DRIVE_DIR / name, encoding='utf-8') if l.strip()]
    print(f'{name}: {len(part)} utterances')
    rows.extend(part)

for r in rows:
    r['audio_filepath'] = str(AUDIO_DIR / Path(r['audio_filepath']).name)
missing = [r['audio_filepath'] for r in rows if not Path(r['audio_filepath']).exists()]
print('total to transcribe:', len(rows), '(expect 431) | missing audio:', len(missing))

Mounted at /content/drive
copying audio folder from Drive to local disk (fast I/O)...
audio files: 431 (expect 431)
test_v1_16k.json: 127 utterances
train_v1_274.json: 274 utterances
val_v1_30.json: 30 utterances
total to transcribe: 431 (expect 431) | missing audio: 0


In [3]:
from huggingface_hub import HfApi, hf_hub_download
import nemo.collections.asr as nemo_asr

REPO = 'ilprl-docse/NwachaMuna-NepConformer-Aug'
files = HfApi().list_repo_files(REPO)
print('repo files:', files)
nemo_files = [f for f in files if f.endswith('.nemo')]
assert nemo_files, f'No .nemo file found in {REPO}'
local_path = hf_hub_download(repo_id=REPO, filename=nemo_files[0])
model = nemo_asr.models.ASRModel.restore_from(local_path)
model.eval()
print('model loaded:', type(model).__name__)

repo files: ['.gitattributes', 'NwachaMuna-NepConformer-Aug-config.json', 'NwachaMuna-NepConformer-Aug.nemo', 'README.md']


NwachaMuna-NepConformer-Aug.nemo: reconstructing file:   0%|          |  0.00B /  123MB            

NwachaMuna-NepConformer-Aug.nemo: downloading bytes:           |  0.00B            

[NeMo I 2026-10-19 22:15:06 mixins:194] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-10-19 22:15:06 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_train.json
    sample_rate: 16000
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-10-19 22:15:06 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/mono_final_nemo_val.json
    sample_rate: 16000
    batch_size: 4
    shuffle: false
    use

[NeMo I 2026-10-19 22:15:07 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--ilprl-docse--NwachaMuna-NepConformer-Aug/snapshots/eba3d554805ae275fba9f0cda6374e15c94b0e7f/NwachaMuna-NepConformer-Aug.nemo.
model loaded: EncDecCTCModelBPE


In [4]:
import time

BATCH = 16
hyps = []
t0 = time.time()
for s in range(0, len(rows), BATCH):
    chunk = [r['audio_filepath'] for r in rows[s:s + BATCH]]
    out = model.transcribe(chunk, batch_size=BATCH, verbose=False)
    for item in out:
        if isinstance(item, str):
            hyps.append(item)
        elif hasattr(item, 'text'):
            hyps.append(item.text)
        elif isinstance(item, (list, tuple)) and item:
            first = item[0]
            hyps.append(first if isinstance(first, str) else getattr(first, 'text', ''))
        else:
            hyps.append('')
    print(f'{min(s + BATCH, len(rows))}/{len(rows)} transcribed ({time.time() - t0:.0f}s)', flush=True)

assert len(hyps) == len(rows)
print('all done:', len(hyps), 'hypotheses')

[NeMo W 2026-10-19 22:15:29 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-19 22:15:29 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


16/431 transcribed (6s)


[NeMo W 2026-10-19 22:15:35 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-19 22:15:35 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


32/431 transcribed (6s)


[NeMo W 2026-10-19 22:15:35 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-19 22:15:35 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


...

400/431 transcribed (12s)


[NeMo W 2026-10-19 22:15:41 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-19 22:15:41 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


416/431 transcribed (13s)


[NeMo W 2026-10-19 22:15:42 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-19 22:15:42 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


431/431 transcribed (13s)
all done: 431 hypotheses


In [7]:
import re, unicodedata
from collections import defaultdict
import jiwer

# EXACT same convention as T1 / RESULTS.md v1.0:
# NFC; strip danda + ? . , ! ; : + ZWJ/ZWNJ; collapse whitespace;
# CER with spaces kept (jiwer default), corpus-level weighted.
PUNCT = re.compile(r'[\u0964?.!,;:\u200c\u200d]')

def norm(t):
    t = unicodedata.normalize('NFC', str(t))
    t = PUNCT.sub(' ', t)
    return re.sub(r'\s+', ' ', t).strip()

agg = defaultdict(lambda: [0.0, 0.0, 0])   # speaker -> [edits, chars, n_utts]
for r, h in zip(rows, hyps):
    ref_n = norm(r['text'])
    err = jiwer.cer(ref_n, norm(h)) * len(ref_n)
    a = agg[r.get('speaker', '?')]
    a[0] += err
    a[1] += len(ref_n)
    a[2] += 1

print('Quick per-speaker preview - corpus CER (spaces kept, paper convention):')
for spk in sorted(agg):
    e, n, k = agg[spk]
    print(f'  {spk}: n={k:3d}   CER={100 * e / n:6.2f}%')
print()
print('spk04 should land close to the paper headline 26.75% (small decoding drift possible).')

Quick per-speaker preview - corpus CER (spaces kept, paper convention):
  spk01: n= 49   CER= 30.83%
  spk02: n=101   CER= 26.77%
  spk03: n= 51   CER= 29.51%
  spk04: n=127   CER= 26.91%
  spk05: n= 48   CER= 28.38%
  spk06: n= 22   CER= 28.37%
  spk07: n= 33   CER= 31.41%

spk04 should land close to the paper headline 26.75% (small decoding drift possible).


In [6]:
!pip install -q jiwer==4.0.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.0 MB/s eta 0:00:00


In [8]:
import csv
from pathlib import Path

out_path = DRIVE_DIR / 'transcriptions_all431_v1.tsv'
with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['id', 'speaker', 'domain', 'reference', 'hypothesis'])
    for r, h in zip(rows, hyps):
        uid = Path(r['audio_filepath']).stem
        w.writerow([uid, r.get('speaker', ''), r.get('domain', ''), r['text'], h])

print('saved:', out_path)
print()
print('NEXT (on your Mac):')
print('  1. download transcriptions_all431_v1.tsv from Drive to ~/Desktop/experiments/')
print('  2. python eval_newari.py tsv --tsv ~/Desktop/experiments/transcriptions_all431_v1.tsv --out scores_all431.csv')
print('  3. paste the summary back -> per-speaker table for the paper comes next')

saved: /content/drive/MyDrive/experiments/transcriptions_all431_v1.tsv

NEXT (on your Mac):
  1. download transcriptions_all431_v1.tsv from Drive to ~/Desktop/experiments/
  2. python eval_newari.py tsv --tsv ~/Desktop/experiments/transcriptions_all431_v1.tsv --out scores_all431.csv
  3. paste the summary back -> per-speaker table for the paper comes next


In [9]:
!pip install -q "nemo_toolkit[asr]" "huggingface_hub" "jiwer==4.0.0" 2>&1 | tail -5
print("install done")

install done


In [10]:
import json, zipfile, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/experiments')
AUDIO_DIR = Path('/content/audio_16k')

zip_path = DRIVE_DIR / 'audio_16k.zip'
if AUDIO_DIR.exists() and len(list(AUDIO_DIR.glob('*.wav'))) > 0:
    print('audio already local, reusing')
elif zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content/')
    print('unzipped', zip_path.name)
elif (DRIVE_DIR / 'audio_16k').exists():
    print('copying audio from Drive to local disk...')
    shutil.copytree(DRIVE_DIR / 'audio_16k', AUDIO_DIR, dirs_exist_ok=True)
else:
    raise SystemExit('No audio found. Expected audio_16k.zip or audio_16k/ in Drive/experiments')

if len(list(AUDIO_DIR.glob('*.wav'))) == 0:
    stray = list(Path('/content').glob('audio_*.wav'))
    if stray:
        AUDIO_DIR.mkdir(exist_ok=True)
        for f in stray:
            shutil.move(str(f), str(AUDIO_DIR / f.name))

rows = [json.loads(l) for l in open(DRIVE_DIR / 'test_v1_16k.json', encoding='utf-8') if l.strip()]
print('test manifest:', len(rows), '(expect 127)')
assert len(rows) == 127

for r in rows:
    r['audio_filepath'] = str(AUDIO_DIR / Path(r['audio_filepath']).name)
missing = [r['audio_filepath'] for r in rows if not Path(r['audio_filepath']).exists()]
print('missing audio:', len(missing))
assert not missing

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
audio already local, reusing
test manifest: 127 (expect 127)
missing audio: 0


In [11]:
from huggingface_hub import HfApi, hf_hub_download
import nemo.collections.asr as nemo_asr

REPO = 'ilprl-docse/NwachaMuna-NepConformer-FT'
files = HfApi().list_repo_files(REPO)
nemo_files = [f for f in files if f.endswith('.nemo')]
assert nemo_files, f'No .nemo file in {REPO}'
local_path = hf_hub_download(repo_id=REPO, filename=nemo_files[0])
model = nemo_asr.models.ASRModel.restore_from(local_path)
model.eval()
print('loaded', REPO)

NwachaMuna-NepConformer-FT.nemo: reconstructing file:   0%|          |  0.00B /  123MB            

NwachaMuna-NepConformer-FT.nemo: downloading bytes:           |  0.00B            

[NeMo I 2026-10-20 01:53:50 mixins:194] Tokenizer SentencePieceTokenizer initialized with 128 tokens


[NeMo W 2026-10-20 01:53:50 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/mono_nemo_train.json
    sample_rate: 16000
    batch_size: 4
    shuffle: true
    num_workers: 4
    pin_memory: true
    max_duration: 16.7
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-10-20 01:53:50 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/mono_nemo_val.json
    sample_rate: 16000
    batch_size: 4
    shuffle: false
    use_start_end_t

[NeMo I 2026-10-20 01:53:50 save_restore_connector:287] Model EncDecCTCModelBPE was successfully restored from /root/.cache/huggingface/hub/models--ilprl-docse--NwachaMuna-NepConformer-FT/snapshots/7a8c566fdb8ab907eab41f017d45f82449597fc5/NwachaMuna-NepConformer-FT.nemo.
loaded ilprl-docse/NwachaMuna-NepConformer-FT


In [12]:
import time
BATCH = 16
hyps = []
t0 = time.time()
for s in range(0, len(rows), BATCH):
    chunk = [r['audio_filepath'] for r in rows[s:s + BATCH]]
    out = model.transcribe(chunk, batch_size=BATCH, verbose=False)
    for item in out:
        if isinstance(item, str):
            hyps.append(item)
        elif hasattr(item, 'text'):
            hyps.append(item.text)
        elif isinstance(item, (list, tuple)) and item:
            first = item[0]
            hyps.append(first if isinstance(first, str) else getattr(first, 'text', ''))
        else:
            hyps.append('')
    print(f'{min(s + BATCH, len(rows))}/{len(rows)} done ({time.time() - t0:.0f}s)', flush=True)
assert len(hyps) == len(rows)

[NeMo W 2026-10-20 01:54:18 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-20 01:54:18 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


16/127 done (0s)


[NeMo W 2026-10-20 01:54:18 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-20 01:54:18 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


32/127 done (0s)


[NeMo W 2026-10-20 01:54:19 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-20 01:54:19 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


...

96/127 done (1s)


[NeMo W 2026-10-20 01:54:19 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-20 01:54:19 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


112/127 done (1s)


[NeMo W 2026-10-20 01:54:19 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-10-20 01:54:19 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


127/127 done (1s)


In [13]:
import re, unicodedata, csv, jiwer

PUNCT = re.compile(r'[\u0964?.!,;:\u200c\u200d]')
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    text = PUNCT.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()

tot_e = tot_l = 0.0
for r, h in zip(rows, hyps):
    ref_n = normalize(r['text'])
    tot_e += jiwer.cer(ref_n, normalize(h)) * len(ref_n)
    tot_l += len(ref_n)
print(f'FT model CER (spaces kept): {100 * tot_e / tot_l:.2f}%   (your Aug model: 26.75%)')

out_path = DRIVE_DIR / 'scores_ft_test127.csv'
with open(out_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['id', 'ref_norm', 'hyp_norm'])
    for r, h in zip(rows, hyps):
        w.writerow([Path(r['audio_filepath']).stem, normalize(r['text']), normalize(h)])
print('saved:', out_path)

FT model CER (spaces kept): 30.04%   (your Aug model: 26.75%)
saved: /content/drive/MyDrive/experiments/scores_ft_test127.csv


In [21]:
!pip -q uninstall -y transformers
!pip -q install "transformers==4.49.0" "accelerate>=1.3" bitsandbytes sacrebleu sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.2 MB/s eta 0:00:00


In [22]:
from google.colab import drive
import csv, os

drive.mount('/content/drive')

# edit if your folder differs — file must be in YOUR Drive
PROBE_TSV = "/content/drive/MyDrive/experiments/llm_probe_30.tsv"
assert os.path.exists(PROBE_TSV), f"not found: {PROBE_TSV} — upload llm_probe_30.tsv there"

rows = []
with open(PROBE_TSV, newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        rows.append(r)
assert len(rows) == 30, f"expected 30 rows, got {len(rows)}"
for r in rows:
    assert r["newar_sentence"].strip() and r["nepali_sentence"].strip()
print(f"loaded {len(rows)} rows OK")

PROMPT_TRANSLATE = ("Translate the following Newari text into Nepali. "
                    "Output only the translation, nothing else.")
PROMPT_COPY = ("Copy the following text exactly, character for character. "
               "Output only the copy, nothing else.")
MAX_NEW_TOKENS = 128

TASKS = []
for r in rows:
    nw, ne = r["newar_sentence"].strip(), r["nepali_sentence"].strip()
    TASKS.append(("translate",   PROMPT_TRANSLATE, nw, ne))
    TASKS.append(("echo_newari", PROMPT_COPY,      nw, nw))
    TASKS.append(("echo_nepali", PROMPT_COPY,      ne, ne))
print(f"{len(TASKS)} generations per model")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loaded 30 rows OK
90 generations per model


In [23]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sacrebleu import corpus_chrf, sentence_chrf

RESULTS = []

def run_model(model_id, tag):
    print(f"\n===== {tag} =====")
    tok = AutoTokenizer.from_pretrained(model_id)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
    model.eval()

    hyps = {"translate": [], "echo_newari": [], "echo_nepali": []}
    refs = {"translate": [], "echo_newari": [], "echo_nepali": []}

    for i, (cond, instr, src, ref) in enumerate(TASKS):
        messages = [{"role": "user", "content": f"{instr}\n\n{src}"}]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        hyp = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        hyps[cond].append(hyp); refs[cond].append(ref)
        RESULTS.append({"model": tag, "condition": cond, "id": rows[i % 30]["id"],
                        "source": src, "ref": ref, "hyp": hyp,
                        "chrf_item": round(sentence_chrf(hyp, [ref]).score, 1)})
        if (i + 1) % 30 == 0:
            print(f"  {i+1}/{len(TASKS)} done")

    print(f"--- {tag} corpus chrF ---")
    for cond in ("translate", "echo_newari", "echo_nepali"):
        print(f"  {cond:12s}: {corpus_chrf(hyps[cond], [refs[cond]]).score:6.1f}")

    del model, tok
    gc.collect(); torch.cuda.empty_cache()

In [24]:
run_model("Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct")


===== Qwen2.5-7B-Instruct =====


[transformers] `layer_type_validation` is deprecated and will be removed in v5.20. Use `PreTrainedConfig.validate_layer_type` instead


ModuleNotFoundError: No module named 'transformers.models.audioflamingo3'

In [1]:
!pip -q uninstall -y transformers
!pip -q install "transformers==4.49.0" "accelerate>=1.3" bitsandbytes sacrebleu sentencepiece

In [2]:
import transformers, torch, accelerate, bitsandbytes, sacrebleu
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4."), "run Step 1 again, then Restart session"
print("OK — now run Cells 2-6")

transformers 4.49.0
OK — now run Cells 2-6


In [3]:
from google.colab import drive
import csv, os

drive.mount('/content/drive')

# edit if your folder differs — file must be in YOUR Drive
PROBE_TSV = "/content/drive/MyDrive/experiments/llm_probe_30.tsv"
assert os.path.exists(PROBE_TSV), f"not found: {PROBE_TSV} — upload llm_probe_30.tsv there"

rows = []
with open(PROBE_TSV, newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f, delimiter="\t"):
        rows.append(r)
assert len(rows) == 30, f"expected 30 rows, got {len(rows)}"
for r in rows:
    assert r["newar_sentence"].strip() and r["nepali_sentence"].strip()
print(f"loaded {len(rows)} rows OK")

PROMPT_TRANSLATE = ("Translate the following Newari text into Nepali. "
                    "Output only the translation, nothing else.")
PROMPT_COPY = ("Copy the following text exactly, character for character. "
               "Output only the copy, nothing else.")
MAX_NEW_TOKENS = 128

TASKS = []
for r in rows:
    nw, ne = r["newar_sentence"].strip(), r["nepali_sentence"].strip()
    TASKS.append(("translate",   PROMPT_TRANSLATE, nw, ne))
    TASKS.append(("echo_newari", PROMPT_COPY,      nw, nw))
    TASKS.append(("echo_nepali", PROMPT_COPY,      ne, ne))
print(f"{len(TASKS)} generations per model")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
loaded 30 rows OK
90 generations per model


In [4]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sacrebleu import corpus_chrf, sentence_chrf

RESULTS = []

def run_model(model_id, tag):
    print(f"\n===== {tag} =====")
    tok = AutoTokenizer.from_pretrained(model_id)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
    model.eval()

    hyps = {"translate": [], "echo_newari": [], "echo_nepali": []}
    refs = {"translate": [], "echo_newari": [], "echo_nepali": []}

    for i, (cond, instr, src, ref) in enumerate(TASKS):
        messages = [{"role": "user", "content": f"{instr}\n\n{src}"}]
        text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        hyp = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        hyps[cond].append(hyp); refs[cond].append(ref)
        RESULTS.append({"model": tag, "condition": cond, "id": rows[i % 30]["id"],
                        "source": src, "ref": ref, "hyp": hyp,
                        "chrf_item": round(sentence_chrf(hyp, [ref]).score, 1)})
        if (i + 1) % 30 == 0:
            print(f"  {i+1}/{len(TASKS)} done")

    print(f"--- {tag} corpus chrF ---")
    for cond in ("translate", "echo_newari", "echo_nepali"):
        print(f"  {cond:12s}: {corpus_chrf(hyps[cond], [refs[cond]]).score:6.1f}")

    del model, tok
    gc.collect(); torch.cuda.empty_cache()

In [5]:
run_model("Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct")


===== Qwen2.5-7B-Instruct =====


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  30/90 done
  60/90 done
  90/90 done
--- Qwen2.5-7B-Instruct corpus chrF ---
  translate   :   22.3
  echo_newari :   98.2
  echo_nepali :   99.5


In [6]:
run_model("unsloth/Mistral-7B-Instruct-v0.3", "Mistral-7B-Instruct-v0.3")


===== Mistral-7B-Instruct-v0.3 =====


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

  30/90 done
  60/90 done
  90/90 done
--- Mistral-7B-Instruct-v0.3 corpus chrF ---
  translate   :   13.1
  echo_newari :   73.3
  echo_nepali :   89.0


In [7]:
# Cell 5b — frontier model via Google AI Studio (free tier)
!pip -q install google-genai
import time
from getpass import getpass
from google import genai

client = genai.Client(api_key=getpass("Paste your Google AI Studio key: "))
API_MODEL = "gemini-2.5-flash"

done = 0
for i, (cond, instr, src, ref) in enumerate(TASKS):
    hyp = ""
    for attempt in range(4):
        try:
            resp = client.models.generate_content(
                model=API_MODEL,
                contents=f"{instr}\n\n{src}",
                config={"temperature": 0})
            hyp = (resp.text or "").strip()
            break
        except Exception as e:
            print(f"  retry {attempt+1}: {type(e).__name__}"); time.sleep(10)
    RESULTS.append({"model": API_MODEL, "condition": cond, "id": rows[i % 30]["id"],
                    "source": src, "ref": ref, "hyp": hyp,
                    "chrf_item": round(sentence_chrf(hyp, [ref]).score, 1)})
    done += 1
    if done % 30 == 0: print(f"  {done}/90 done")
    time.sleep(3)   # respect free-tier rate limits

hyps = {c: [] for c in ("translate", "echo_newari", "echo_nepali")}
refs = {c: [] for c in ("translate", "echo_newari", "echo_nepali")}
for r in RESULTS:
    if r["model"] == API_MODEL:
        hyps[r["condition"]].append(r["hyp"]); refs[r["condition"]].append(r["ref"])
print(f"--- {API_MODEL} corpus chrF ---")
for cond in hyps:
    print(f"  {cond:12s}: {corpus_chrf(hyps[cond], [refs[cond]]).score:6.1f}")

  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: ClientError
  retry 1: ClientError
  retry 2: ClientError
  retry 3: ClientError
  retry 4: 

KeyboardInterrupt: 

In [8]:
from getpass import getpass
from google import genai

client = genai.Client(api_key=getpass("Paste your key again (carefully, no spaces): "))

for m in ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash"]:
    try:
        resp = client.models.generate_content(model=m, contents="Say OK")
        print(f"\nWORKS with {m}:", resp.text)
        break
    except Exception as e:
        print(f"\n{m}: {type(e).__name__}")
        print(str(e)[:300])

Paste your key again (carefully, no spaces): ··········

gemini-2.5-flash: ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/d

gemini-2.0-flash: ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-start

gemini-1.5-flash: ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 

In [9]:
RESULTS = [r for r in RESULTS if not str(r["model"]).startswith("gemini")]
print("rows now:", len(RESULTS))   # expect 180

rows now: 180


In [10]:
API_MODEL = "gemini-3.8-flash"

In [ ]:
# Cell 5b — frontier model via Google AI Studio (free tier)
!pip -q install google-genai
import time
from getpass import getpass
from google import genai

client = genai.Client(api_key=getpass("Paste your Google AI Studio key: "))
API_MODEL = "gemini-3.8-flash"

done = 0
for i, (cond, instr, src, ref) in enumerate(TASKS):
    hyp = ""
    for attempt in range(4):
        try:
            resp = client.models.generate_content(
                model=API_MODEL,
                contents=f"{instr}\n\n{src}",
                config={"temperature": 0})
            hyp = (resp.text or "").strip()
            break
        except Exception as e:
            print(f"  retry {attempt+1}: {type(e).__name__}"); time.sleep(10)
    RESULTS.append({"model": API_MODEL, "condition": cond, "id": rows[i % 30]["id"],
                    "source": src, "ref": ref, "hyp": hyp,
                    "chrf_item": round(sentence_chrf(hyp, [ref]).score, 1)})
    done += 1
    if done % 30 == 0: print(f"  {done}/90 done")
    time.sleep(3)   # respect free-tier rate limits

hyps = {c: [] for c in ("translate", "echo_newari", "echo_nepali")}
refs = {c: [] for c in ("translate", "echo_newari", "echo_nepali")}
for r in RESULTS:
    if r["model"] == API_MODEL:
        hyps[r["condition"]].append(r["hyp"]); refs[r["condition"]].append(r["ref"])
print(f"--- {API_MODEL} corpus chrF ---")
for cond in hyps:
    print(f"  {cond:12s}: {corpus_chrf(hyps[cond], [refs[cond]]).score:6.1f}")

Paste your Google AI Studio key: ··········
  retry 1: ServerError
  retry 2: ServerError
  retry 1: ServerError


In [4]:
# ════════════════════════════════════════════════════════════════════
# Cell 5b v2 — Gemini probe (30 × 3 = 90 calls), quota-aware
#   • clean QUOTA STOP at the free-tier daily cap (progress saved, resumes)
#   • one retry when a thinking model returns empty text
#   • 429 short-rate-limits → backoff; day-cap 429 → stop, don't burn quota
# ════════════════════════════════════════════════════════════════════
import os, csv, re, time, glob, unicodedata

# ── CONFIG ──────────────────────────────────────────────────────────
API_MODEL     = "gemini-3.1-flash-lite"  # ← set from Cell 5e's recommendation!
                                      # (gemini-3.8-flash works but = 20 req/day free)
PROBE_PATH    = ""
RESULTS_CSV   = "/content/gemini_probe_results.csv"
SLEEP_BETWEEN = 5.0
MAX_RETRIES   = 6

# Prompt templates — MUST stay identical to your Qwen / Mistral probe cell.
PROMPTS = {
    "translate":   ("Translate the following Newari sentence into Nepali. "
                    "Output only the Nepali translation, nothing else.\n\n{src}"),
    "echo_newari": ("Repeat the following text exactly as it is. "
                    "Output only the text itself, nothing else.\n\n{src}"),
    "echo_nepali": ("Repeat the following text exactly as it is. "
                    "Output only the text itself, nothing else.\n\n{src}"),
}
CONDITIONS = ["translate", "echo_newari", "echo_nepali"]

# ── 1. load the 30 probe sentences ──────────────────────────────────
import pandas as pd

def _pick_col(cols, candidates):
    low = {str(c).lower().strip(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    for c_low, c_orig in low.items():
        for cand in candidates:
            if cand in c_low:
                return c_orig
    return None

probe = None
for var in ("probe_df", "df_probe", "probe", "probe_items", "probe30"):
    obj = globals().get(var)
    if obj is None:
        continue
    if hasattr(obj, "columns"):
        probe = obj.copy()
        print(f"[data] using notebook variable `{var}`  ({len(probe)} rows)")
        break
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        probe = pd.DataFrame(obj)
        print(f"[data] built DataFrame from list variable `{var}`")
        break
if probe is None and PROBE_PATH:
    probe = pd.read_csv(PROBE_PATH)
    print(f"[data] loaded {len(probe)} rows from PROBE_PATH")
if probe is None:
    for path in ("/content/probe_sentences.csv", "/content/probe30.csv",
                 "/content/t6_probe.csv", "/content/probe_items.csv",
                 "/content/probe_sentences.tsv", "/content/probe30.tsv"):
        if os.path.exists(path):
            probe = pd.read_csv(path, sep="\t" if path.endswith(".tsv") else ",")
            print(f"[data] loaded {len(probe)} rows from {path}")
            break
if probe is None:
    raise SystemExit(
        "[data] probe sentences not found.\n"
        "  → set PROBE_PATH at the top to the exact CSV/TSV the Qwen / Mistral\n"
        "    probe used (the SAME 30 rows), then re-run this cell."
    )

c_new = _pick_col(probe.columns, ["newari", "newari_text", "new", "source_newari", "src_newari", "src", "source"])
c_nep = _pick_col(probe.columns, ["nepali", "nepali_text", "nep", "nepali_ref", "ref_nepali", "reference_nepali", "ref", "reference", "tgt", "target"])
c_id  = _pick_col(probe.columns, ["utt_id", "uttid", "id", "idx", "index"])
assert c_new and c_nep, f"[data] cannot recognise columns in: {list(probe.columns)}"
assert c_new != c_nep, f"[data] both languages resolved to the same column '{c_new}'"
probe = probe.dropna(subset=[c_new, c_nep]).reset_index(drop=True)
if len(probe) != 30:
    print(f"[data] WARNING: expected 30 probe sentences, found {len(probe)}")
if c_id is None:
    probe["utt_id"] = probe.index.astype(str)
    c_id = "utt_id"
print(f"[data] {len(probe)} sentences | newari='{c_new}'  nepali='{c_nep}'  id='{c_id}'")

# ── 2. client + model ───────────────────────────────────────────────
from google import genai

_api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if _api_key is None:
    try:
        from google.colab import userdata
        _api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
if "client" not in globals() or globals()["client"] is None:
    if not _api_key:
        _api_key = getpass("Paste your Google AI Studio key: ") if False else None
    if not _api_key:
        raise SystemExit("[auth] no API key found — run Cell 5e first (it sets the key + client).")
    client = genai.Client(api_key=_api_key)

def parse_delay(text):
    m = re.search(r"retry in\s+(?:(\d+)h)?(?:(\d+)m)?([\d.]+)s", text)
    if m:
        h, mi, s = m.groups()
        return int(h or 0) * 3600 + int(mi or 0) * 60 + float(s or 0)
    m = re.search(r"retryDelay':\s*'(\d+)s", text)
    return float(m.group(1)) if m else None

class QuotaStop(Exception):
    def __init__(self, model, delay_s, message):
        super().__init__(message)
        self.model, self.delay_s = model, delay_s

def _ping(model):
    client.models.generate_content(model=model, contents="ping",
                                   config={"temperature": 0, "max_output_tokens": 5})

def resolve_model():
    global API_MODEL
    try:
        _ping(API_MODEL)
        print(f"[model] {API_MODEL} → OK")
        return
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
            d = parse_delay(msg) or 0
            print(f"[model] {API_MODEL} → reachable, but today's quota is used up "
                  f"(resets in ~{d/3600:.1f}h). Keeping it — the run will QUOTA-STOP cleanly.")
            return
        print(f"[model] {API_MODEL} → FAILED\n         {msg}")
    print("[model] listing models visible to this key ...")
    avail = []
    try:
        for m in client.models.list():
            avail.append(m.name.replace("models/", ""))
    except Exception as e:
        raise SystemExit(f"[model] cannot list models → {type(e).__name__}: {e}")
    prefer = ["gemini-3.1-flash-lite", "gemini-3-flash-preview",
              "gemini-3.1-flash-lite-preview", "gemini-3.8-flash",
              "gemini-flash-latest", "gemini-flash-lite-latest"]
    for p in prefer:
        match = next((n for n in avail if n == p or n.startswith(p) or p in n), None)
        if match:
            try:
                _ping(match)
                API_MODEL = match
                print(f"[model] fallback → {API_MODEL} works  (pin it in CONFIG)")
                return
            except Exception as e:
                msg = f"{type(e).__name__}: {e}"
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
                    continue          # dry bucket ≠ dead model, try next name
                print(f"[model] {match} ping failed → {msg[:120]}")
    raise SystemExit("[model] no usable model found. Visible flash models:\n  "
                     + "\n  ".join(n for n in sorted(avail) if "flash" in n)[:1500])

# ── 3. robust call + scorer ─────────────────────────────────────────
def call_gemini(prompt):
    last = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = client.models.generate_content(model=API_MODEL, contents=prompt,
                                               config={"temperature": 0})
            txt = (r.text or "").strip()
            if txt:
                return txt
            if attempt < MAX_RETRIES:
                print("    [empty output → retrying once (thinking models sometimes do this)]")
                time.sleep(2)
                continue
            return ""
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            up = msg.upper()
            if any(k in up for k in ("API_KEY_INVALID", "PERMISSION_DENIED",
                                     "LOCATION IS NOT SUPPORTED", "USER LOCATION")):
                raise RuntimeError("unrecoverable API error → " + msg) from e
            if "429" in msg or "RESOURCE_EXHAUSTED" in up:
                d = parse_delay(msg) or 0
                if d > 900:                      # daily cap → stop, save, come back later
                    raise QuotaStop(API_MODEL, d, msg) from e
                # short per-minute limit → backoff and retry
            last = e
            wait = min(60, 4 * 2 ** (attempt - 1))
            print(f"    [retry {attempt}/{MAX_RETRIES}] {msg[:140]}  → wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {MAX_RETRIES} retries → {type(last).__name__}: {last}")

def _lev(a, b):
    if a == b: return 0
    if not a:  return len(b)
    if not b:  return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def score(hyp, ref):
    """char-level similarity 0–100 — keep IDENTICAL to the Qwen/Mistral cell."""
    hyp = unicodedata.normalize("NFC", " ".join(str(hyp).split()))
    ref = unicodedata.normalize("NFC", " ".join(str(ref).split()))
    if not hyp:
        return 0.0
    return round(100.0 * (1.0 - _lev(hyp, ref) / max(len(ref), len(hyp))), 2)

# ── 4. resumable run ────────────────────────────────────────────────
FIELDS = ["utt_id", "condition", "model", "input_text", "reference", "output", "score"]
fresh_start, done, other_model_rows = True, set(), 0
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if rows and rows[0] == FIELDS:
        fresh_start = False
        for r in rows[1:]:
            if len(r) == len(FIELDS):
                done.add((r[0], r[1], r[2]))
    else:
        os.replace(RESULTS_CSV, RESULTS_CSV + ".bak")
        print(f"[resume] old CSV has a different schema → moved to {RESULTS_CSV}.bak")

resolve_model()
done = {(u, c) for (u, c, m) in done if m == API_MODEL}
if done:
    print(f"[resume] {len(done)} calls already saved for {API_MODEL} → skipped")

out_f = open(RESULTS_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=FIELDS)
if fresh_start:
    writer.writeheader()

total = len(probe) * len(CONDITIONS)
print(f"\n[run] model={API_MODEL} | {len(probe)} sentences × {len(CONDITIONS)} conditions "
      f"= {total} calls ({total - len(done)} to go)\n")

t0, n_new, quota_stopped = time.time(), 0, None
stop = False
for _, row in probe.iterrows():
    if stop:
        break
    utt_id = str(row[c_id]); newari = str(row[c_new]).strip(); nepali = str(row[c_nep]).strip()
    for cond in CONDITIONS:
        if (utt_id, cond) in done:
            continue
        src, ref = {"translate":   (newari, nepali),
                    "echo_newari": (newari, newari),
                    "echo_nepali": (nepali, nepali)}[cond]
        try:
            out = call_gemini(PROMPTS[cond].format(src=src))
        except QuotaStop as q:
            quota_stopped = q
            stop = True
            break
        except Exception as e:
            out_f.close()
            print(f"\n[FATAL] utt {utt_id} / {cond} → {type(e).__name__}: {e}")
            print("All finished calls are saved — fix and re-run this cell.")
            raise
        s = score(out, ref)
        writer.writerow({"utt_id": utt_id, "condition": cond, "model": API_MODEL,
                         "input_text": src, "reference": ref, "output": out, "score": s})
        out_f.flush(); n_new += 1
        print(f"  {utt_id:>8} {cond:<12} score={s:6.2f}")
        time.sleep(SLEEP_BETWEEN)
out_f.close()
print(f"\n[done] {n_new} new calls in {time.time() - t0:.0f}s")

if quota_stopped:
    q = quota_stopped
    print(f"\n[QUOTA STOP] free-tier daily cap reached for {q.model} (resets in ~{q.delay_s/3600:.1f}h).")
    print("  Progress is saved — re-run this SAME cell after the reset and it resumes.")
    print("  Options: (a) re-run daily until 90/90;  (b) enable billing (90 calls ≈ pennies);")
    print("           (c) second Google project = fresh quota;  (d) ask me for a batched variant (9-18 calls).")

# ── 5. summary ──────────────────────────────────────────────────────
res = pd.read_csv(RESULTS_CSV)
res = res[res["model"] == API_MODEL]
print("\n" + "=" * 54)
print(f"GEMINI PROBE — {API_MODEL}   (n={len(res)} of {total} calls)"
      + ("   [PARTIAL — quota stop]" if quota_stopped else ""))
print("=" * 54)
if len(res) == 0:
    print("no rows saved for this model yet")
else:
    for cond in CONDITIONS:
        sub = res[res["condition"] == cond]
        mean = sub["score"].mean() if len(sub) else float("nan")
        print(f"{cond:<14} mean score = {mean:6.2f}   (n={len(sub)})")
    print("=" * 54)
    print("line it up with:  Qwen2.5-7B  22.3 / 98.2 / 99.5")
    print("                  Mistral-7B  13.1 / 73.3 / 89.0")

# ── 6. combined T6 table ────────────────────────────────────────────
parts = []
for path in sorted(glob.glob("/content/*_probe_results.csv")):
    dfp = pd.read_csv(path)
    if not {"condition", "score"}.issubset(dfp.columns) or not len(dfp):
        continue
    if "model" in dfp.columns and dfp["model"].nunique() > 1:
        dfp = dfp[dfp["model"] == dfp["model"].iloc[-1]]
    name = os.path.basename(path).replace("_probe_results.csv", "")
    parts.append((name, dfp.groupby("condition")["score"].mean()))
if parts:
    order = {"qwen": 0, "mistral": 1, "gemini": 2}
    parts.sort(key=lambda t: order.get(t[0].lower(), 9))
    print("\n=== T6 probe — combined (mean score per condition) ===")
    print(f"{'model':<16}" + "".join(f"{c:>13}" for c in CONDITIONS))
    for name, ser in parts:
        print(f"{name:<16}" + "".join(f"{ser.get(c, float('nan')):>13.2f}" for c in CONDITIONS))

SystemExit: [data] probe sentences not found.
  → set PROBE_PATH at the top to the exact CSV/TSV the Qwen / Mistral
    probe used (the SAME 30 rows), then re-run this cell.

In [2]:
# Cell 5e — find a Gemini model with AVAILABLE daily quota (1 test call each)
import os, re
from getpass import getpass
from google import genai

if "client" not in globals() or globals()["client"] is None:
    key = os.environ.get("GOOGLE_API_KEY") or getpass("Paste your Google AI Studio key: ")
    os.environ["GOOGLE_API_KEY"] = key
    client = genai.Client(api_key=key)

def parse_delay(text):
    m = re.search(r"retry in\s+(?:(\d+)h)?(?:(\d+)m)?([\d.]+)s", text)
    if m:
        h, mi, s = m.groups()
        return int(h or 0) * 3600 + int(mi or 0) * 60 + float(s or 0)
    m = re.search(r"retryDelay':\s*'(\d+)s", text)
    return float(m.group(1)) if m else None

def quota_limit(text):
    m = re.search(r"limit:\s*(\d+),\s*model:\s*(\S+)", text)
    return (int(m.group(1)), m.group(2)) if m else (None, None)

print("STEP 1: full flash list (in case something interesting is hiding past the first 10)")
try:
    flash = [m.name.replace("models/", "") for m in client.models.list() if "flash" in m.name]
    print("\n".join(flash))
except Exception as e:
    flash = []
    print(f"list FAILED — {type(e).__name__}: {e!r}")

CANDIDATES = ["gemini-3.1-flash-lite", "gemini-3-flash-preview",
              "gemini-3.1-flash-lite-preview", "gemini-2.5-flash-lite"]
CANDIDATES += [n for n in flash if ("3.1-flash" in n or "3-flash" in n)
               and "image" not in n and "tts" not in n and n not in CANDIDATES][:2]

print("\nSTEP 2: one real generate call per candidate")
status = {}
for m in CANDIDATES:
    try:
        r = client.models.generate_content(model=m, contents="Reply with exactly: OK",
                                           config={"temperature": 0, "max_output_tokens": 300})
        cands = getattr(r, "candidates", None) or []
        fr = getattr(cands[0], "finish_reason", "?") if cands else "?"
        print(f"OK       {m:<34} text={(r.text or '').strip()!r}  finish={fr}")
        status[m] = "OK"
    except Exception as e:
        msg = str(e)
        if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
            lim, mdl = quota_limit(msg)
            d = parse_delay(msg) or 0
            print(f"QUOTA    {m:<34} daily limit={lim}  resets in ~{d/3600:.1f}h")
            status[m] = f"quota-dry (limit {lim})"
        elif "404" in msg:
            print(f"DEAD     {m:<34} not available to this key (404)")
            status[m] = "unavailable"
        else:
            print(f"ERROR    {m:<34} {type(e).__name__}: {msg[:120]}")
            status[m] = "error"

print("\n=== RECOMMENDATION ===")
ok_models = [m for m, s in status.items() if s == "OK"]
if ok_models:
    pick = "gemini-3.1-flash-lite" if "gemini-3.1-flash-lite" in ok_models else ok_models[0]
    print(f'>>> set in Cell 5b:  API_MODEL = "{pick}"   then run 5b')
    print("    if its cap is also 20/day, 5b now stops cleanly at the cap (QUOTA STOP),")
    print("    saves progress, and you re-run the same cell after reset — it resumes.")
else:
    print('no model has quota right now → in ~3h the 3.8-flash bucket resets; then run 5b with')
    print('    API_MODEL = "gemini-3.8-flash"  (20/day → continue over several days, resumable),')
    print("    or enable billing on the project, or ask me for a batched-prompt variant (9-18 calls total).")

STEP 1: full flash list (in case something interesting is hiding past the first 10)
gemini-2.5-flash
gemini-2.5-flash-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
gemini-3.1-flash-lite-preview
gemini-3.1-flash-lite
gemini-3.1-flash-image-preview
gemini-3.1-flash-image
gemini-3.1-flash-lite-image
gemini-3.5-flash
gemini-3.5-flash-lite
gemini-omni-flash-preview
gemini-omni-1.1-flash
gemini-3.6-flash
gemini-3.7-flash
gemini-3.8-flash
gemini-3.1-flash-tts-preview
gemini-3.8-flash-tts
gemini-3.8-flash-lite-tts
gemini-2.5-flash-native-audio-latest
gemini-2.5-flash-native-audio-preview-09-2025
gemini-2.5-flash-native-audio-preview-12-2025
gemini-3.1-flash-live-preview

STEP 2: one real generate call per candidate
OK       gemini-3.1-flash-lite              text='OK'  finish=FinishReason.STOP
OK       gemini-3-flash-preview             text='OK'  finish=FinishReason.STOP
OK       gemini-3.1-flash-lite-preview      

In [5]:
# ════════════════════════════════════════════════════════════════════
# Cell 5b v2 — Gemini probe (30 × 3 = 90 calls), quota-aware
#   • clean QUOTA STOP at the free-tier daily cap (progress saved, resumes)
#   • one retry when a thinking model returns empty text
#   • 429 short-rate-limits → backoff; day-cap 429 → stop, don't burn quota
# ════════════════════════════════════════════════════════════════════
import os, csv, re, time, glob, unicodedata

# ── CONFIG ──────────────────────────────────────────────────────────
API_MODEL     = "gemini-3.1-flash-lite"  # ← set from Cell 5e's recommendation!
                                      # (gemini-3.8-flash works but = 20 req/day free)
PROBE_PATH    = ""
RESULTS_CSV   = "/content/gemini_probe_results.csv"
SLEEP_BETWEEN = 5.0
MAX_RETRIES   = 6

# Prompt templates — MUST stay identical to your Qwen / Mistral probe cell.
PROMPTS = {
    "translate":   ("Translate the following Newari sentence into Nepali. "
                    "Output only the Nepali translation, nothing else.\n\n{src}"),
    "echo_newari": ("Repeat the following text exactly as it is. "
                    "Output only the text itself, nothing else.\n\n{src}"),
    "echo_nepali": ("Repeat the following text exactly as it is. "
                    "Output only the text itself, nothing else.\n\n{src}"),
}
CONDITIONS = ["translate", "echo_newari", "echo_nepali"]

# ── 1. load the 30 probe sentences ──────────────────────────────────
import pandas as pd

def _pick_col(cols, candidates):
    low = {str(c).lower().strip(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    for c_low, c_orig in low.items():
        for cand in candidates:
            if cand in c_low:
                return c_orig
    return None

probe = None
for var in ("probe_df", "df_probe", "probe", "probe_items", "probe30"):
    obj = globals().get(var)
    if obj is None:
        continue
    if hasattr(obj, "columns"):
        probe = obj.copy()
        print(f"[data] using notebook variable `{var}`  ({len(probe)} rows)")
        break
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        probe = pd.DataFrame(obj)
        print(f"[data] built DataFrame from list variable `{var}`")
        break
if probe is None and PROBE_PATH:
    probe = pd.read_csv(PROBE_PATH)
    print(f"[data] loaded {len(probe)} rows from PROBE_PATH")
if probe is None:
    for path in ("/content/probe_sentences.csv", "/content/probe30.csv",
                 "/content/t6_probe.csv", "/content/probe_items.csv",
                 "/content/probe_sentences.tsv", "/content/probe30.tsv"):
        if os.path.exists(path):
            probe = pd.read_csv(path, sep="\t" if path.endswith(".tsv") else ",")
            print(f"[data] loaded {len(probe)} rows from {path}")
            break
if probe is None:
    raise SystemExit(
        "[data] probe sentences not found.\n"
        "  → set PROBE_PATH at the top to the exact CSV/TSV the Qwen / Mistral\n"
        "    probe used (the SAME 30 rows), then re-run this cell."
    )

c_new = _pick_col(probe.columns, ["newari", "newari_text", "new", "source_newari", "src_newari", "src", "source"])
c_nep = _pick_col(probe.columns, ["nepali", "nepali_text", "nep", "nepali_ref", "ref_nepali", "reference_nepali", "ref", "reference", "tgt", "target"])
c_id  = _pick_col(probe.columns, ["utt_id", "uttid", "id", "idx", "index"])
assert c_new and c_nep, f"[data] cannot recognise columns in: {list(probe.columns)}"
assert c_new != c_nep, f"[data] both languages resolved to the same column '{c_new}'"
probe = probe.dropna(subset=[c_new, c_nep]).reset_index(drop=True)
if len(probe) != 30:
    print(f"[data] WARNING: expected 30 probe sentences, found {len(probe)}")
if c_id is None:
    probe["utt_id"] = probe.index.astype(str)
    c_id = "utt_id"
print(f"[data] {len(probe)} sentences | newari='{c_new}'  nepali='{c_nep}'  id='{c_id}'")

# ── 2. client + model ───────────────────────────────────────────────
from google import genai

_api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if _api_key is None:
    try:
        from google.colab import userdata
        _api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
if "client" not in globals() or globals()["client"] is None:
    if not _api_key:
        _api_key = getpass("Paste your Google AI Studio key: ") if False else None
    if not _api_key:
        raise SystemExit("[auth] no API key found — run Cell 5e first (it sets the key + client).")
    client = genai.Client(api_key=_api_key)

def parse_delay(text):
    m = re.search(r"retry in\s+(?:(\d+)h)?(?:(\d+)m)?([\d.]+)s", text)
    if m:
        h, mi, s = m.groups()
        return int(h or 0) * 3600 + int(mi or 0) * 60 + float(s or 0)
    m = re.search(r"retryDelay':\s*'(\d+)s", text)
    return float(m.group(1)) if m else None

class QuotaStop(Exception):
    def __init__(self, model, delay_s, message):
        super().__init__(message)
        self.model, self.delay_s = model, delay_s

def _ping(model):
    client.models.generate_content(model=model, contents="ping",
                                   config={"temperature": 0, "max_output_tokens": 5})

def resolve_model():
    global API_MODEL
    try:
        _ping(API_MODEL)
        print(f"[model] {API_MODEL} → OK")
        return
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
            d = parse_delay(msg) or 0
            print(f"[model] {API_MODEL} → reachable, but today's quota is used up "
                  f"(resets in ~{d/3600:.1f}h). Keeping it — the run will QUOTA-STOP cleanly.")
            return
        print(f"[model] {API_MODEL} → FAILED\n         {msg}")
    print("[model] listing models visible to this key ...")
    avail = []
    try:
        for m in client.models.list():
            avail.append(m.name.replace("models/", ""))
    except Exception as e:
        raise SystemExit(f"[model] cannot list models → {type(e).__name__}: {e}")
    prefer = ["gemini-3.1-flash-lite", "gemini-3-flash-preview",
              "gemini-3.1-flash-lite-preview", "gemini-3.8-flash",
              "gemini-flash-latest", "gemini-flash-lite-latest"]
    for p in prefer:
        match = next((n for n in avail if n == p or n.startswith(p) or p in n), None)
        if match:
            try:
                _ping(match)
                API_MODEL = match
                print(f"[model] fallback → {API_MODEL} works  (pin it in CONFIG)")
                return
            except Exception as e:
                msg = f"{type(e).__name__}: {e}"
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
                    continue          # dry bucket ≠ dead model, try next name
                print(f"[model] {match} ping failed → {msg[:120]}")
    raise SystemExit("[model] no usable model found. Visible flash models:\n  "
                     + "\n  ".join(n for n in sorted(avail) if "flash" in n)[:1500])

# ── 3. robust call + scorer ─────────────────────────────────────────
def call_gemini(prompt):
    last = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = client.models.generate_content(model=API_MODEL, contents=prompt,
                                               config={"temperature": 0})
            txt = (r.text or "").strip()
            if txt:
                return txt
            if attempt < MAX_RETRIES:
                print("    [empty output → retrying once (thinking models sometimes do this)]")
                time.sleep(2)
                continue
            return ""
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            up = msg.upper()
            if any(k in up for k in ("API_KEY_INVALID", "PERMISSION_DENIED",
                                     "LOCATION IS NOT SUPPORTED", "USER LOCATION")):
                raise RuntimeError("unrecoverable API error → " + msg) from e
            if "429" in msg or "RESOURCE_EXHAUSTED" in up:
                d = parse_delay(msg) or 0
                if d > 900:                      # daily cap → stop, save, come back later
                    raise QuotaStop(API_MODEL, d, msg) from e
                # short per-minute limit → backoff and retry
            last = e
            wait = min(60, 4 * 2 ** (attempt - 1))
            print(f"    [retry {attempt}/{MAX_RETRIES}] {msg[:140]}  → wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {MAX_RETRIES} retries → {type(last).__name__}: {last}")

def _lev(a, b):
    if a == b: return 0
    if not a:  return len(b)
    if not b:  return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def score(hyp, ref):
    """char-level similarity 0–100 — keep IDENTICAL to the Qwen/Mistral cell."""
    hyp = unicodedata.normalize("NFC", " ".join(str(hyp).split()))
    ref = unicodedata.normalize("NFC", " ".join(str(ref).split()))
    if not hyp:
        return 0.0
    return round(100.0 * (1.0 - _lev(hyp, ref) / max(len(ref), len(hyp))), 2)

# ── 4. resumable run ────────────────────────────────────────────────
FIELDS = ["utt_id", "condition", "model", "input_text", "reference", "output", "score"]
fresh_start, done, other_model_rows = True, set(), 0
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if rows and rows[0] == FIELDS:
        fresh_start = False
        for r in rows[1:]:
            if len(r) == len(FIELDS):
                done.add((r[0], r[1], r[2]))
    else:
        os.replace(RESULTS_CSV, RESULTS_CSV + ".bak")
        print(f"[resume] old CSV has a different schema → moved to {RESULTS_CSV}.bak")

resolve_model()
done = {(u, c) for (u, c, m) in done if m == API_MODEL}
if done:
    print(f"[resume] {len(done)} calls already saved for {API_MODEL} → skipped")

out_f = open(RESULTS_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=FIELDS)
if fresh_start:
    writer.writeheader()

total = len(probe) * len(CONDITIONS)
print(f"\n[run] model={API_MODEL} | {len(probe)} sentences × {len(CONDITIONS)} conditions "
      f"= {total} calls ({total - len(done)} to go)\n")

t0, n_new, quota_stopped = time.time(), 0, None
stop = False
for _, row in probe.iterrows():
    if stop:
        break
    utt_id = str(row[c_id]); newari = str(row[c_new]).strip(); nepali = str(row[c_nep]).strip()
    for cond in CONDITIONS:
        if (utt_id, cond) in done:
            continue
        src, ref = {"translate":   (newari, nepali),
                    "echo_newari": (newari, newari),
                    "echo_nepali": (nepali, nepali)}[cond]
        try:
            out = call_gemini(PROMPTS[cond].format(src=src))
        except QuotaStop as q:
            quota_stopped = q
            stop = True
            break
        except Exception as e:
            out_f.close()
            print(f"\n[FATAL] utt {utt_id} / {cond} → {type(e).__name__}: {e}")
            print("All finished calls are saved — fix and re-run this cell.")
            raise
        s = score(out, ref)
        writer.writerow({"utt_id": utt_id, "condition": cond, "model": API_MODEL,
                         "input_text": src, "reference": ref, "output": out, "score": s})
        out_f.flush(); n_new += 1
        print(f"  {utt_id:>8} {cond:<12} score={s:6.2f}")
        time.sleep(SLEEP_BETWEEN)
out_f.close()
print(f"\n[done] {n_new} new calls in {time.time() - t0:.0f}s")

if quota_stopped:
    q = quota_stopped
    print(f"\n[QUOTA STOP] free-tier daily cap reached for {q.model} (resets in ~{q.delay_s/3600:.1f}h).")
    print("  Progress is saved — re-run this SAME cell after the reset and it resumes.")
    print("  Options: (a) re-run daily until 90/90;  (b) enable billing (90 calls ≈ pennies);")
    print("           (c) second Google project = fresh quota;  (d) ask me for a batched variant (9-18 calls).")

# ── 5. summary ──────────────────────────────────────────────────────
res = pd.read_csv(RESULTS_CSV)
res = res[res["model"] == API_MODEL]
print("\n" + "=" * 54)
print(f"GEMINI PROBE — {API_MODEL}   (n={len(res)} of {total} calls)"
      + ("   [PARTIAL — quota stop]" if quota_stopped else ""))
print("=" * 54)
if len(res) == 0:
    print("no rows saved for this model yet")
else:
    for cond in CONDITIONS:
        sub = res[res["condition"] == cond]
        mean = sub["score"].mean() if len(sub) else float("nan")
        print(f"{cond:<14} mean score = {mean:6.2f}   (n={len(sub)})")
    print("=" * 54)
    print("line it up with:  Qwen2.5-7B  22.3 / 98.2 / 99.5")
    print("                  Mistral-7B  13.1 / 73.3 / 89.0")

# ── 6. combined T6 table ────────────────────────────────────────────
parts = []
for path in sorted(glob.glob("/content/*_probe_results.csv")):
    dfp = pd.read_csv(path)
    if not {"condition", "score"}.issubset(dfp.columns) or not len(dfp):
        continue
    if "model" in dfp.columns and dfp["model"].nunique() > 1:
        dfp = dfp[dfp["model"] == dfp["model"].iloc[-1]]
    name = os.path.basename(path).replace("_probe_results.csv", "")
    parts.append((name, dfp.groupby("condition")["score"].mean()))
if parts:
    order = {"qwen": 0, "mistral": 1, "gemini": 2}
    parts.sort(key=lambda t: order.get(t[0].lower(), 9))
    print("\n=== T6 probe — combined (mean score per condition) ===")
    print(f"{'model':<16}" + "".join(f"{c:>13}" for c in CONDITIONS))
    for name, ser in parts:
        print(f"{name:<16}" + "".join(f"{ser.get(c, float('nan')):>13.2f}" for c in CONDITIONS))

SystemExit: [data] probe sentences not found.
  → set PROBE_PATH at the top to the exact CSV/TSV the Qwen / Mistral
    probe used (the SAME 30 rows), then re-run this cell.

In [6]:
# Cell 5-prep — reload llm_probe_30.tsv on the fresh CPU runtime + sacrebleu
import os, glob

OPTION = "upload"        # "upload" (easiest — file is on your machine)  or  "drive"

if OPTION == "upload":
    from google.colab import files
    print("→ pick  llm_probe_30.tsv  (it's in ~/Desktop/experiments/ on your computer)")
    up = files.upload()
    PROBE_PATH = list(up.keys())[0]
else:
    assert os.path.exists("/content/drive/MyDrive"), "mount Drive first: from google.colab import drive; drive.mount('/content/drive')"
    hits = (glob.glob("/content/drive/MyDrive/llm_probe_30.tsv")
            + glob.glob("/content/drive/MyDrive/*/llm_probe_30.tsv")
            + glob.glob("/content/drive/MyDrive/*/*/llm_probe_30.tsv"))
    assert hits, "not found at those Drive levels — set PROBE_PATH manually"
    PROBE_PATH = hits[0]

import pandas as pd
probe_df = pd.read_csv(PROBE_PATH, sep="\t")
assert {"newar_sentence", "nepali_sentence"} <= set(probe_df.columns), f"bad columns: {list(probe_df.columns)}"
assert len(probe_df) == 30, f"expected 30 rows, got {len(probe_df)}"
probe_df = probe_df.reset_index(drop=True)
print(probe_df[["id", "newar_sentence", "nepali_sentence"]].head(2))
print(f"[prep] 30 probe rows ready from {PROBE_PATH}")

!pip -q install sacrebleu
import sacrebleu
print(f"[prep] sacrebleu {sacrebleu.__version__} ready (chrF scorer)")

→ pick  llm_probe_30.tsv  (it's in ~/Desktop/experiments/ on your computer)


Saving llm_probe_30.tsv to llm_probe_30.tsv
   id                              newar_sentence  \
0  58  गाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?   
1  13       मेगु पसलय नं थुलि दे विचाच्वंगु दइचा।   

                        nepali_sentence  
0  गाइड साहेब, आज कति घण्टा हिँड्ने हो?  
1         अर्को पसलले यतिकै दिँदो रहेछ।  
[prep] 30 probe rows ready from llm_probe_30.tsv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 6.7 MB/s eta 0:00:00
[prep] sacrebleu 2.6.0 ready (chrF scorer)


In [7]:
# ════════════════════════════════════════════════════════════════════
# Cell 5b v3 — Gemini probe, quota-aware, CHRF-SCORED (matches Qwen/Mistral)
#   run Cell 5-prep FIRST (defines probe_df + sacrebleu)
# ════════════════════════════════════════════════════════════════════
_pre = {n: globals()[n] for n in ("PROMPTS", "TASKS", "PROMPT_TEMPLATES",
                                  "prompt_templates", "prompts")
        if n in globals() and isinstance(globals()[n], dict)}

import os, csv, re, time, glob, unicodedata

API_MODEL     = "gemini-3.1-flash-lite"   # ← Cell 5e's pick
PROBE_PATH    = globals().get("PROBE_PATH", "")
RESULTS_CSV   = "/content/gemini_probe_results.csv"
SLEEP_BETWEEN = 5.0
MAX_RETRIES   = 6
CONDITIONS    = ["translate", "echo_newari", "echo_nepali"]

try:
    import sacrebleu
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"], check=True)
    import sacrebleu

def _norm_keys(d):
    return {str(k).strip().lower(): v for k, v in d.items()}

PROMPTS = None
for _name, _d in _pre.items():
    _nd = _norm_keys(_d)
    if set(CONDITIONS).issubset(_nd.keys()):
        PROMPTS = {c: _nd[c] for c in CONDITIONS}
        print(f"[prompts] ADOPTED from notebook variable `{_name}` (your original Cell 2 wording)")
        break
if PROMPTS is None:
    PROMPTS = {
        "translate":   ("Translate the following Newari sentence into Nepali. "
                        "Output only the Nepali translation, nothing else.\n\n{src}"),
        "echo_newari": ("Repeat the following text exactly as it is. "
                        "Output only the text itself, nothing else.\n\n{src}"),
        "echo_nepali": ("Repeat the following text exactly as it is. "
                        "Output only the text itself, nothing else.\n\n{src}"),
    }
    print("[prompts] using built-in templates — if your Cell 2 wording differs, "
          "re-run Cell 2 before this cell, or edit PROMPTS here to match it!")

_PLACEHOLDERS = ("src", "text", "sentence", "newar", "newar_sentence", "input", "x")
def build_prompt(template, src):
    t = str(template)
    for ph in _PLACEHOLDERS:
        try:
            p = t.format(**{ph: src})
            if src in p:
                return p
        except Exception:
            continue
    print("    [prompt] no placeholder found — appending source after template")
    return t + "\n\n" + src

# ── data (probe_df from Cell 5-prep, or fall back to PROBE_PATH / known files) ──
import pandas as pd

probe = None
for var in ("probe_df", "df_probe", "probe", "probe_items"):
    obj = globals().get(var)
    if obj is not None and hasattr(obj, "columns"):
        probe = obj.copy()
        print(f"[data] using notebook variable `{var}`  ({len(probe)} rows)")
        break
if probe is None and PROBE_PATH and os.path.exists(PROBE_PATH):
    probe = pd.read_csv(PROBE_PATH, sep="\t" if PROBE_PATH.endswith(".tsv") else ",")
if probe is None:
    raise SystemExit("[data] run Cell 5-prep first (it loads llm_probe_30.tsv).")

def _pick_col(cols, candidates):
    low = {str(c).lower().strip(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    for c_low, c_orig in low.items():
        for cand in candidates:
            if cand in c_low:
                return c_orig
    return None

c_new = _pick_col(probe.columns, ["newar_sentence", "newari", "newari_text", "new", "src", "source"])
c_nep = _pick_col(probe.columns, ["nepali_sentence", "nepali", "nepali_text", "nep", "ref", "reference", "tgt"])
c_id  = _pick_col(probe.columns, ["id", "utt_id", "uttid", "idx", "index"])
assert c_new and c_nep, f"[data] cannot recognise columns in: {list(probe.columns)}"
assert c_new != c_nep,  f"[data] both languages resolved to '{c_new}'"
probe = probe.dropna(subset=[c_new, c_nep]).reset_index(drop=True)
if len(probe) != 30:
    print(f"[data] WARNING: expected 30 probe sentences, found {len(probe)}")
if c_id is None:
    probe["utt_id"] = probe.index.astype(str); c_id = "utt_id"
print(f"[data] {len(probe)} rows | newari='{c_new}'  nepali='{c_nep}'  id='{c_id}'")

# ── client + model ──────────────────────────────────────────────────
from google import genai

_api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if _api_key is None:
    try:
        from google.colab import userdata
        _api_key = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
if "client" not in globals() or globals()["client"] is None:
    assert _api_key, "[auth] run Cell 5e first (it sets the key + client)."
    client = genai.Client(api_key=_api_key)

def parse_delay(text):
    m = re.search(r"retry in\s+(?:(\d+)h)?(?:(\d+)m)?([\d.]+)s", text)
    if m:
        h, mi, s = m.groups()
        return int(h or 0) * 3600 + int(mi or 0) * 60 + float(s or 0)
    m = re.search(r"retryDelay':\s*'(\d+)s", text)
    return float(m.group(1)) if m else None

class QuotaStop(Exception):
    def __init__(self, model, delay_s, message):
        super().__init__(message)
        self.model, self.delay_s = model, delay_s

def _ping(model):
    client.models.generate_content(model=model, contents="ping",
                                   config={"temperature": 0, "max_output_tokens": 5})

def resolve_model():
    global API_MODEL
    try:
        _ping(API_MODEL)
        print(f"[model] {API_MODEL} → OK")
        return
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
            d = parse_delay(msg) or 0
            print(f"[model] {API_MODEL} quota dry right now (resets ~{d/3600:.1f}h) — keeping it")
            return
        print(f"[model] {API_MODEL} → FAILED\n         {msg}")
    avail = []
    try:
        for m in client.models.list():
            avail.append(m.name.replace("models/", ""))
    except Exception as e:
        raise SystemExit(f"[model] cannot list models → {type(e).__name__}: {e}")
    for p in ["gemini-3.1-flash-lite", "gemini-3-flash-preview",
              "gemini-3.1-flash-lite-preview", "gemini-3.8-flash"]:
        match = next((n for n in avail if n == p or n.startswith(p) or p in n), None)
        if match:
            try:
                _ping(match)
                API_MODEL = match
                print(f"[model] fallback → {API_MODEL}")
                return
            except Exception as e:
                msg = f"{type(e).__name__}: {e}"
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg.upper():
                    continue
    raise SystemExit("[model] no usable model — re-run Cell 5e.")

# ── call + chrF scorer ──────────────────────────────────────────────
def call_gemini(prompt):
    widened, last = False, None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            cfg = ({"temperature": 0, "max_output_tokens": 1024} if widened
                   else {"temperature": 0, "max_output_tokens": 256})
            r = client.models.generate_content(model=API_MODEL, contents=prompt, config=cfg)
            txt = (r.text or "").strip()
            if txt:
                return txt
            if not widened:
                widened = True
                print("    [empty output → retrying with larger token cap]")
                continue
            if attempt < MAX_RETRIES:
                time.sleep(2); continue
            return ""
        except Exception as e:
            msg = f"{type(e).__name__}: {e}"
            up = msg.upper()
            if any(k in up for k in ("API_KEY_INVALID", "PERMISSION_DENIED",
                                     "LOCATION IS NOT SUPPORTED", "USER LOCATION")):
                raise RuntimeError("unrecoverable API error → " + msg) from e
            if "429" in msg or "RESOURCE_EXHAUSTED" in up:
                d = parse_delay(msg) or 0
                if d > 900:
                    raise QuotaStop(API_MODEL, d, msg) from e
            last = e
            wait = min(60, 4 * 2 ** (attempt - 1))
            print(f"    [retry {attempt}/{MAX_RETRIES}] {msg[:140]}  → wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {MAX_RETRIES} retries → {type(last).__name__}: {last}")

def score(hyp, ref):
    """sacrebleu sentence chrF (0–100) — SAME metric as the Qwen/Mistral runs."""
    hyp = " ".join(str(hyp).split())
    ref = " ".join(str(ref).split())
    if not hyp:
        return 0.0
    return round(sacrebleu.sentence_chrf(hyp, [ref]).score, 2)

# ── resumable run ───────────────────────────────────────────────────
FIELDS = ["utt_id", "condition", "model", "input_text", "reference", "output", "score"]
fresh_start, done = True, set()
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    if rows and rows[0] == FIELDS:
        fresh_start = False
        for r in rows[1:]:
            if len(r) == len(FIELDS):
                done.add((r[0], r[1], r[2]))
    else:
        os.replace(RESULTS_CSV, RESULTS_CSV + ".bak")
        print(f"[resume] old CSV had a different schema → {RESULTS_CSV}.bak")

resolve_model()
done = {(u, c) for (u, c, m) in done if m == API_MODEL}
if done:
    print(f"[resume] {len(done)} calls already saved for {API_MODEL} → skipped")

print("\n[prompt check] first prompt per condition (truncated):")
_r0 = probe.iloc[0]
for _c in CONDITIONS:
    _src = str(_r0[c_new]).strip() if _c in ("translate", "echo_newari") else str(_r0[c_nep]).strip()
    print(f"  {_c:<12} {build_prompt(PROMPTS[_c], _src)[:140]!r}")

out_f = open(RESULTS_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=FIELDS)
if fresh_start:
    writer.writeheader()

total = len(probe) * len(CONDITIONS)
print(f"\n[run] model={API_MODEL} | greedy (temp 0), max_output_tokens 256 "
      f"(mirrors local 128-new-token cap) | {total} calls ({total - len(done)} to go)\n")

t0, n_new, quota_stopped, stop = time.time(), 0, None, False
for _, row in probe.iterrows():
    if stop:
        break
    utt_id = str(row[c_id]); newari = str(row[c_new]).strip(); nepali = str(row[c_nep]).strip()
    for cond in CONDITIONS:
        if (utt_id, cond) in done:
            continue
        src, ref = {"translate":   (newari, nepali),
                    "echo_newari": (newari, newari),
                    "echo_nepali": (nepali, nepali)}[cond]
        try:
            out = call_gemini(build_prompt(PROMPTS[cond], src))
        except QuotaStop as q:
            quota_stopped, stop = q, True
            break
        except Exception as e:
            out_f.close()
            print(f"\n[FATAL] utt {utt_id} / {cond} → {type(e).__name__}: {e}")
            print("Progress saved — fix and re-run this cell.")
            raise
        s = score(out, ref)
        writer.writerow({"utt_id": utt_id, "condition": cond, "model": API_MODEL,
                         "input_text": src, "reference": ref, "output": out, "score": s})
        out_f.flush(); n_new += 1
        print(f"  {utt_id:>8} {cond:<12} chrF={s:6.2f}")
        time.sleep(SLEEP_BETWEEN)
out_f.close()
print(f"\n[done] {n_new} new calls in {time.time() - t0:.0f}s")

if quota_stopped:
    q = quota_stopped
    print(f"\n[QUOTA STOP] daily cap reached for {q.model} (resets ~{q.delay_s/3600:.1f}h). "
          "Progress saved — re-run this cell after reset (resumes), or use options b/c/d.")

# ── summary (chrF, comparable with Qwen/Mistral) ────────────────────
res = pd.read_csv(RESULTS_CSV)
res = res[res["model"] == API_MODEL]
print("\n" + "=" * 54)
print(f"GEMINI PROBE — {API_MODEL}   (n={len(res)} of {total} calls)"
      + ("   [PARTIAL — quota stop]" if quota_stopped else ""))
print("=" * 54)
if len(res) == 0:
    print("no rows saved for this model yet")
else:
    for cond in CONDITIONS:
        sub = res[res["condition"] == cond]
        mean = sub["score"].mean() if len(sub) else float("nan")
        print(f"{cond:<14} chrF = {mean:6.2f}   (n={len(sub)})")
    print("=" * 54)
    print("compare:  Qwen2.5-7B  22.3 / 98.2 / 99.5")
    print("          Mistral-7B  13.1 / 73.3 / 89.0")

# ── combined table (tolerant to your llm_probe_results.csv schema) ──
parts = []
for path in sorted(glob.glob("/content/*_probe_results.csv")):
    dfp = pd.read_csv(path)
    low = {c.lower(): c for c in dfp.columns}
    cond_col  = next((low[k] for k in ("condition", "task") if k in low), None)
    score_col = next((low[k] for k in ("score", "chrf", "sentence_chrf") if k in low), None)
    if not (cond_col and score_col) or not len(dfp):
        continue
    if "model" in dfp.columns and dfp["model"].nunique() > 1:
        dfp = dfp[dfp["model"] == dfp["model"].iloc[-1]]
    name = os.path.basename(path).replace("_probe_results.csv", "")
    parts.append((name, dfp.groupby(cond_col)[score_col].mean()))
if parts:
    order = {"qwen": 0, "mistral": 1, "gemini": 2}
    parts.sort(key=lambda t: order.get(t[0].lower(), 9))
    print("\n=== T6 probe — combined (mean chrF per condition) ===")
    print(f"{'model':<16}" + "".join(f"{c:>13}" for c in CONDITIONS))
    for name, ser in parts:
        print(f"{name:<16}" + "".join(f"{ser.get(c, float('nan')):>13.2f}" for c in CONDITIONS))

[prompts] ADOPTED from notebook variable `PROMPTS` (your original Cell 2 wording)
[data] using notebook variable `probe_df`  (30 rows)
[data] 30 rows | newari='newar_sentence'  nepali='nepali_sentence'  id='id'
[model] gemini-3.1-flash-lite → OK

[prompt check] first prompt per condition (truncated):
  translate    'Translate the following Newari sentence into Nepali. Output only the Nepali translation, nothing else.\n\nगाइड साहेब, थौ गुलि घण्टा न्यासि वने'
  echo_newari  'Repeat the following text exactly as it is. Output only the text itself, nothing else.\n\nगाइड साहेब, थौ गुलि घण्टा न्यासि वनेगु खः?'
  echo_nepali  'Repeat the following text exactly as it is. Output only the text itself, nothing else.\n\nगाइड साहेब, आज कति घण्टा हिँड्ने हो?'

[run] model=gemini-3.1-flash-lite | greedy (temp 0), max_output_tokens 256 (mirrors local 128-new-token cap) | 90 calls (90 to go)

        58 translate    chrF= 92.73
        58 echo_newari  chrF=100.00
        58 echo_nepali  chrF=100.00
   

In [8]:
# ============================================================
# CELL 6 — T6 finalize: backup + validate + combined paper table
# Run AFTER Cell 5b v3 shows "[done] 90 new calls"
# ============================================================
import os, datetime, pandas as pd

GEMINI_CSV  = "/content/gemini_probe_results.csv"
MODEL_NAME  = "gemini-3.1-flash-lite"   # exact string for citation
ACCESS_DATE = "2026-10-20"              # <-- change if you ran 5b yesterday

# ---------- 0) BACKUP FIRST (fresh runtimes wipe /content!) ----------
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    import shutil
    dst = f"/content/drive/MyDrive/experiments/gemini_probe_results_{datetime.date.today()}.csv"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(GEMINI_CSV, dst)
    print(f"[backup] copied -> {dst}")
except Exception as e:
    print(f"[backup] Drive unavailable ({type(e).__name__}) -> browser download instead")
    from google.colab import files
    files.download(GEMINI_CSV)

# ---------- 1) Load + validate ----------
g = pd.read_csv(GEMINI_CSV)
g["output"] = g["output"].astype(str)
g["score"]  = pd.to_numeric(g["score"], errors="coerce")
assert {"utt_id","condition","model","output","score"} <= set(g.columns), f"bad schema: {list(g.columns)}"
assert g["model"].nunique() == 1, "mixed models in CSV!"
assert g.duplicated(["utt_id","condition"]).sum() == 0, "duplicate (utt_id,condition) rows!"
n_per = g.groupby("condition").size().to_dict()
assert all(v == 30 for v in n_per.values()), f"expected 30/condition, got {n_per}"

per_cond = g.groupby("condition")["score"].agg(["count","mean","min","max"]).round(2)
print(per_cond)
for c in ["echo_newari","echo_nepali"]:
    assert per_cond.loc[c,"mean"] >= 99.0, f"{c} below ceiling?!"
print("[ok] 90 rows | 30 per condition | single model | echo at ceiling")

# ---------- 2) Combined T6 table ----------
# Qwen/Mistral means are banked from the original llm_probe_results.csv
# (verified in the earlier session; not present on this fresh runtime)
BANKED = {
    "Qwen2.5-7B": {"translate": 22.3, "echo_newari": 98.2, "echo_nepali": 99.5},
    "Mistral-7B": {"translate": 13.1, "echo_newari": 73.3, "echo_nepali": 89.0},
}
rows = [{"model": m, **d, "gap_newari": round(d["echo_newari"]-d["translate"],2)}
        for m, d in BANKED.items()]
gm = {c: round(per_cond.loc[c,"mean"],2) for c in ["translate","echo_newari","echo_nepali"]}
rows.append({"model": MODEL_NAME, **gm, "gap_newari": round(gm["echo_newari"]-gm["translate"],2)})

t6 = pd.DataFrame(rows)[["model","translate","echo_newari","echo_nepali","gap_newari"]]
try:
    print("\n=== FINAL T6 (mean sentence chrF, 30 utts/condition) ===")
    print(t6.to_markdown(index=False))
except ImportError:
    print(t6.to_string(index=False))
t6.to_csv("/content/t6_llm_probe_final.csv", index=False)
print("\n[saved] /content/t6_llm_probe_final.csv")

# ---------- 3) Color for §6.2: hardest + easiest translate cases ----------
tr = g[g.condition=="translate"].sort_values("score")
print("\n[for §6.2] 3 hardest translate cases:")
for _, r in tr.head(3).iterrows():
    print(f"  id={r.utt_id} chrF={r.score:.2f}\n    ref: {r.reference}\n    out: {r.output[:80]}")
print("[for §6.2] easiest case:")
r = tr.iloc[-1]
print(f"  id={r.utt_id} chrF={r.score:.2f}\n    ref: {r.reference}\n    out: {r.output[:80]}")

print(f"\n[cite] {MODEL_NAME} | Google AI Studio API (free tier) | temp=0 (greedy) | "
      f"max_output_tokens=256 | sacrebleu sentence chrF | accessed {ACCESS_DATE}")

Mounted at /content/drive
[backup] copied -> /content/drive/MyDrive/experiments/gemini_probe_results_2026-10-20.csv
             count    mean     min     max
condition                                 
echo_nepali     30  100.00  100.00  100.00
echo_newari     30  100.00  100.00  100.00
translate       30   50.17    7.47   92.73
[ok] 90 rows | 30 per condition | single model | echo at ceiling

=== FINAL T6 (mean sentence chrF, 30 utts/condition) ===
| model                 |   translate |   echo_newari |   echo_nepali |   gap_newari |
|:----------------------|------------:|--------------:|--------------:|-------------:|
| Qwen2.5-7B            |       22.3  |          98.2 |          99.5 |        75.9  |
| Mistral-7B            |       13.1  |          73.3 |          89   |        60.2  |
| gemini-3.1-flash-lite |       50.17 |         100   |         100   |        49.83 |

[saved] /content/t6_llm_probe_final.csv

[for §6.2] 3 hardest translate cases:
  id=167 chrF=7.47
    ref: गर्